# DataVine Analytics — Summative Lab
## Classification, Recommendation and Clustering Prototypes with PCA

**Role:** Junior Data Scientist · **Client projects:** 3 · **Datasets:** Wine, Chickwts, USArrests

## Overview — Bottom Line Up Front

**What this is.** Three prototype machine-learning models, built with one standardized workflow (*prepare → standardize → reduce with PCA → model and tune → evaluate → interpret*), that turn small standard datasets into decision support for three clients.

**Bottom line**
* **Wine classifier:** identifies the wine variety from 13 chemical measurements with **93.3 % accuracy on unseen wines (≈ 95 % in nested cross-validation)**. Ready for a **pilot with human review** of low-confidence cases.
* **Feed recommender:** suggests similar feeds with a similarity score. **Casein ↔ sunflower (0.97)** and **linseed ↔ soybean (0.96)** are near-interchangeable; **horsebean has no close substitute**. A convincing **prototype** that needs richer data.
* **Crime-pattern clustering:** finds **two natural groups of states (30 lower-violence, 20 higher-violence)**; K-Means and GMM agree, and GMM flags Missouri, Illinois and Texas as boundary cases. An **exploratory** segmentation framework.
* **Main caveat:** all datasets are small (178 wines, 71 chicks, 50 states), so results are prototypes, not production guarantees.

### Objectives of the models

| Model | What it intends to achieve | How success is judged |
|---|---|---|
| **1. Wine classifier** | Automatically identify wine variety from chemical properties, for inventory management and quality control | Accuracy, per-class precision/recall, cross-validation |
| **2. Feed recommender** | Suggest comparable feeds to a farmer, based on performance (chick weight) | Cosine-similarity score; agreement with an independent weight benchmark |
| **3. Crime clustering** | Discover natural groupings of states by crime pattern, with a measure of uncertainty | Elbow, silhouette, BIC; agreement between K-Means and GMM |

### Key stakeholders and their needs

| Stakeholder | Need | How this work serves it |
|---|---|---|
| **Premium wine distributor** (inventory and quality-control teams) | Fast, consistent variety identification, and knowing *when to double-check* | Tuned k-NN classifier plus a confidence level for routing doubtful wines to an expert |
| **Agricultural supply company** (sales/advisers and their farmer customers) | Substitute feeds for stock-outs or price pressure, with a transparent reason | Similarity-ranked recommendations showing strength of match and expected weight difference |
| **Public-policy research firm** (analysts and policy makers) | A data-driven grouping of regions to target attention, honest about uncertainty | Two-segment view with GMM membership probabilities and a boundary-state watch-list |
| **DataVine supervisor / reviewers** | A reproducible, documented, validated workflow | Standardized steps, fixed random seed, interpretation under every result |

### Client projects at a glance

| # | Client project | Technique | Dataset |
|---|---|---|---|
| 1 | Wine classification system | k-NN + PCA + GridSearchCV | `wine.csv` |
| 2 | Agricultural feed recommendation engine | PCA + cosine similarity | `chickwts.csv` |
| 3 | Regional crime pattern analysis | K-Means + Gaussian Mixture Models (GMM) | `USArrests.csv` |

### How this notebook is organised
Every analytical step has a **main title** (e.g. *Step 1*), **sub-titles** that follow the lab instructions one-to-one, the **code** that answers each instruction, and a **markdown cell directly below each code cell** that explains:

* **What the code does** – in plain language
* **Why it matters** – the reason this step is in the workflow
* **Interpretation** – what the output tells us
* **Decision** – what we do next because of it

### Table of contents
0. Setup
1. Load and prepare datasets
2. k-NN classification on the Wine dataset
3. Recommendation system using the Chickwts dataset
4. Clustering (K-Means and GMM) on the USArrests dataset
5. Conclusions and recommendations for stakeholders

> **Reproducibility:** every random process uses `random_state = 42`, so re-running the notebook gives identical results.

---
# Step 0 — Setup
## 0.1 Import libraries and set global options

In [ ]:
import os, warnings
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (classification_report, accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, silhouette_score, adjusted_rand_score)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook", palette="deep")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)

# Locate the data files (current folder, ./data, or the upload folder)
FILES = ["wine.csv", "chickwts.csv", "USArrests.csv"]
DATA_DIR = next((d for d in [".", "data", "/mnt/user-data/uploads"]
                 if all(os.path.exists(os.path.join(d, f)) for f in FILES)), None)
assert DATA_DIR is not None, "Place wine.csv, chickwts.csv and USArrests.csv next to this notebook."
print("Data folder:", os.path.abspath(DATA_DIR))

---
# Step 1 — Load and Prepare Datasets
*Goal: make sure all three datasets are accessible, clean, standardized and correctly structured for machine learning.*

## 1.1 Load the Wine, Chickwts, and USArrests datasets

In [ ]:
wine    = pd.read_csv(os.path.join(DATA_DIR, "wine.csv"))
chick   = pd.read_csv(os.path.join(DATA_DIR, "chickwts.csv"))
arrests = pd.read_csv(os.path.join(DATA_DIR, "USArrests.csv"), index_col="rownames")
arrests.index.name = "state"          # the first column holds state names -> use it as the row label

for name, df in {"Wine": wine, "Chickwts": chick, "USArrests": arrests}.items():
    print(f"{name:<10} -> {df.shape[0]} rows x {df.shape[1]} columns")

print("\nWine (first 3 rows)");      display(wine.head(3))
print("Chickwts (first 3 rows)");     display(chick.head(3))
print("USArrests (first 3 rows)");    display(arrests.head(3))

**What the code does.** Reads the three CSV files into pandas DataFrames. For USArrests the first column (`rownames`) holds the state names, so it is used as the row index rather than as a feature. It then prints each table's shape and its first three rows.

**Why it matters.** Before any modelling we must be sure that the data loaded correctly: right number of rows, right columns, and identifiers kept out of the feature matrix.

**Interpretation.**
* **Wine:** 178 wines × 14 columns = 13 numeric chemical measurements (alcohol, flavanoids, proline, …) plus the `target` (variety).
* **Chickwts:** 71 chicks × 2 columns = `weight` (grams) and `feed` (type of feed).
* **USArrests:** 50 states × 4 columns = `Murder`, `Assault`, `Rape` (arrests per 100,000 residents) and `UrbanPop` (% urban population).

**Decision.** The structure matches the lab description, so we proceed to quality checks. State names stay as the index so that they can label the clustering charts later.

## 1.2 Inspect the data for missing values and handle inconsistencies

In [ ]:
# ---- (a) Missing values, duplicates and data types --------------------------------
quality = pd.DataFrame({
    "rows":           {"Wine": len(wine), "Chickwts": len(chick), "USArrests": len(arrests)},
    "columns":        {"Wine": wine.shape[1], "Chickwts": chick.shape[1], "USArrests": arrests.shape[1]},
    "missing values": {"Wine": wine.isna().sum().sum(), "Chickwts": chick.isna().sum().sum(), "USArrests": arrests.isna().sum().sum()},
    "duplicate rows": {"Wine": wine.duplicated().sum(), "Chickwts": chick.duplicated().sum(), "USArrests": arrests.duplicated().sum()},
})
display(quality)

# ---- (b) Text consistency: trim spaces / unify case in categorical columns --------
before = chick["feed"].unique().tolist()
chick["feed"]   = chick["feed"].str.strip().str.lower()
arrests.index   = arrests.index.str.strip()
print("\nFeed labels before cleaning:", sorted(before))
print("Feed labels after  cleaning:", sorted(chick["feed"].unique()))
print("Duplicate state names:", arrests.index.duplicated().sum())

# ---- (c) Plausibility / range checks ----------------------------------------------
assert (wine.drop(columns="target") > 0).all().all(),        "Wine: unexpected non-positive value"
assert (chick["weight"] > 0).all(),                          "Chickwts: unexpected non-positive weight"
assert (arrests[["Murder", "Assault", "Rape"]] >= 0).all().all(), "USArrests: negative arrest rate"
assert arrests["UrbanPop"].between(0, 100).all(),            "USArrests: UrbanPop outside 0-100 %"
print("\nRange checks passed for all three datasets.")

# ---- (d) Inspect the single duplicated row in chickwts ----------------------------
print("\nDuplicated rows in chickwts (all copies shown):")
display(chick[chick.duplicated(keep=False)])

# ---- (e) Label values / group sizes ------------------------------------------------
print("Wine target values:", sorted(wine["target"].unique()))
print("\nChicks per feed:"); display(chick["feed"].value_counts().to_frame("n_chicks").T)

**What the code does.** Runs four families of checks: **(a)** counts missing values and duplicate rows; **(b)** trims whitespace and unifies case in text columns; **(c)** asserts that values are physically plausible (no negatives, `UrbanPop` between 0 and 100 %); **(d)** shows the duplicated rows; **(e)** lists the target values and group sizes.

**Why it matters.** Missing values, typos in category names or impossible values silently degrade distance-based algorithms such as k-NN, PCA and K-Means.

**Interpretation.**
* **No missing values** in any dataset, and all range checks pass.
* Feed names were already clean (identical before and after cleaning) and there are no duplicate state names.
* The only inconsistency candidate is **one duplicated row in chickwts**: two soybean chicks that both weighed 248 g.
* The wine target contains the three codes 0, 1, 2; the feed groups contain 10–14 chicks each.

**Decision.**
* No imputation or row deletion is needed.
* The duplicated chickwts row is **kept**: the data are individual chick measurements, and two different chicks can legitimately have the same weight (it is a coincidence, not a double entry). Removing it would also bias the soybean group.
* The text-cleaning code stays in the workflow as a safeguard, so the pipeline would still work on a messier file.

## 1.3 Standardize numerical features to improve model performance

In [ ]:
# Standardize each dataset:  z = (x - mean) / std   ->  mean 0, standard deviation 1
wine_features = wine.drop(columns="target")
wine_scaled   = pd.DataFrame(StandardScaler().fit_transform(wine_features),
                             columns=wine_features.columns, index=wine.index)

chick_scaled = chick.copy()
chick_scaled["weight_z"] = StandardScaler().fit_transform(chick[["weight"]]).ravel()

arrests_scaled = pd.DataFrame(StandardScaler().fit_transform(arrests),
                              columns=arrests.columns, index=arrests.index)

# Verification: spread of each wine feature before vs after scaling
scale_check = pd.DataFrame({"std BEFORE": wine_features.std(), "std AFTER": wine_scaled.std(),
                            "mean AFTER": wine_scaled.mean()})
display(scale_check)
print(f"Wine: the largest feature spread is {wine_features.std().max() / wine_features.std().min():,.0f}x the smallest before scaling.")
print("Chickwts weight_z  -> mean = %.3f, std = %.3f" % (chick_scaled.weight_z.mean(), chick_scaled.weight_z.std(ddof=0)))
print("USArrests (scaled) -> max |mean| = %.3f" % arrests_scaled.mean().abs().max())

# Visual proof
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=wine_features, orient="h", ax=ax[0], fliersize=2);  ax[0].set_title("Wine features — BEFORE scaling"); ax[0].set_xscale("symlog")
sns.boxplot(data=wine_scaled,   orient="h", ax=ax[1], fliersize=2);  ax[1].set_title("Wine features — AFTER scaling")
plt.tight_layout(); plt.show()

**What the code does.** Applies `StandardScaler`, which transforms every numeric column to *z = (x − mean) / standard deviation*, so each feature ends up with mean 0 and standard deviation 1. The wine target is excluded (it is a label, not a feature). A before/after table and box-plots verify the result.

**Why it matters.** k-NN, PCA and K-Means are all driven by distances or variances. Without scaling, the variables with the largest numeric range dominate the results regardless of how informative they are.

**Interpretation.**
* In the raw wine data, `proline` has a standard deviation of ≈315 while `nonflavanoid_phenols` has ≈0.12 – a **2,530× difference**. Left unscaled, proline would swamp all other chemistry.
* After scaling, every feature has mean ≈ 0 and standard deviation ≈ 1.003 (the tiny excess over 1 is only because pandas divides by *n−1* while the scaler divides by *n*).
* Chickwts `weight` and all four USArrests columns are likewise centred and unit-scaled. The box-plots show all wine features now sharing one common scale.

**Decision.**
* These scaled copies are used for exploration and for the unsupervised Steps 3 and 4.
* **For the wine classifier we will *not* reuse this pre-scaled table.** Fitting a scaler on all 178 wines before the train/test split would leak test-set information into training. Instead, scaling becomes the first stage of a `Pipeline` in Step 2, so it is re-fitted on training data only.

## 1.4 Summarize the datasets to confirm their structure

In [ ]:
print("=== WINE: 13 chemical features + target ===")
display(wine.describe().T[["mean", "std", "min", "50%", "max"]])
print("Class balance:"); display(wine["target"].value_counts().sort_index().to_frame("n_wines").assign(share=lambda d: (d.n_wines / d.n_wines.sum()).round(3)).T)

print("\n=== CHICKWTS: weight (g) by feed ===")
feed_summary = chick.groupby("feed")["weight"].agg(["count", "mean", "std", "min", "median", "max"]).sort_values("mean")
display(feed_summary)

print("\n=== USARRESTS: arrests per 100,000 (Murder, Assault, Rape) and % urban population ===")
display(arrests.describe().T[["mean", "std", "min", "50%", "max"]])

fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
sns.countplot(x="target", data=wine, ax=ax[0]);                          ax[0].set_title("Wine — class balance")
order = feed_summary.index
sns.boxplot(data=chick, x="weight", y="feed", order=order, ax=ax[1]);    ax[1].set_title("Chickwts — weight by feed (g)")
sns.heatmap(arrests.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax[2]); ax[2].set_title("USArrests — correlation")
plt.tight_layout(); plt.show()

**What the code does.** Produces descriptive statistics for each dataset, the wine class balance, mean/median weight per feed, and three overview charts (wine class counts, chick weight by feed, USArrests correlation matrix).

**Why it matters.** This confirms that each dataset has the structure and distribution assumed by the models, and reveals issues such as class imbalance or strongly correlated features that influence modelling choices.

**Interpretation.**
* **Wine:** the three classes are reasonably balanced (59 / 71 / 48 wines = 33 % / 40 % / 27 %), so plain accuracy is a fair metric, and no resampling is required. Features sit on very different scales (e.g. proline ≈ 278–1,680 vs. hue ≈ 0.5–1.7), re-confirming the need for scaling.
* **Chickwts:** feeds fall into a clear ladder of average weight – horsebean (160 g) < linseed (219 g) < soybean (246 g) < meatmeal (277 g) < casein (324 g) ≈ sunflower (329 g). Casein and sunflower overlap almost completely, as do (to a lesser degree) linseed and soybean. Horsebean sits alone at the bottom.
* **USArrests:** `Murder`, `Assault` and `Rape` are strongly positively correlated (r = 0.56–0.80), meaning they largely measure the same underlying "violent-crime level". `UrbanPop` is only weakly related to them (r = 0.07–0.41).

**Decision.**
* Wine: use accuracy plus a per-class report; use *stratified* splits to preserve the class proportions.
* Chickwts: since weight is the only performance measure, similarity between feeds will be about similarity of their weight distributions.
* USArrests: the strong correlation among the three crime variables makes PCA an excellent fit, and the weak `UrbanPop` correlation foreshadows the feature-selection decision in Step 4.

---
# Step 2 — k-Nearest Neighbors (k-NN) Classification on the Wine Dataset
*Goal: automatically identify the wine variety from its chemical properties, using PCA to compress the features and GridSearchCV to tune k-NN.*

**Design note — avoiding data leakage.** Scaling and PCA learn from the data (means, variances, directions). If they were fitted on the *whole* dataset before splitting, information from the test wines would leak into training and the accuracy would be over-optimistic. Therefore, from here on, **StandardScaler → PCA → k-NN are chained in one `Pipeline`**, so that during every cross-validation fold and for the final model they are fitted on training data only.

## 2.1 Convert categorical target labels into numeric values

In [ ]:
X = wine.drop(columns="target")
le = LabelEncoder()
y = le.fit_transform(wine["target"])                       # guarantees integer codes 0..n-1
class_names = [f"Class {c}" for c in le.classes_]

print("Original labels  :", list(le.classes_))
print("Encoded labels   :", list(range(len(le.classes_))))
print("Label mapping    :", dict(zip(le.classes_, le.transform(le.classes_))))
print("Encoded class counts:", dict(zip(*np.unique(y, return_counts=True))))

**What the code does.** Separates the features (`X`) from the target and passes the target through `LabelEncoder`, which turns whatever labels exist into consecutive integers 0, 1, 2. It prints the mapping and the class counts, and builds readable class names for later reports.

**Why it matters.** scikit-learn estimators need numeric targets. Using an explicit encoder (instead of assuming the labels are numeric) makes the pipeline safe if the labels ever arrive as text such as *"Barolo"*.

**Interpretation.** The target was already numeric (0, 1, 2), so the mapping is the identity: 0→0, 1→1, 2→2, with 59, 71 and 48 wines respectively.

**Decision.** The encoded vector `y` is used for all modelling. In the reports we refer to the varieties as *Class 0, Class 1, Class 2*, because the file does not tell us the commercial names of the cultivars.

## 2.2 Split the data into training and test sets *(additional step)*

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

print(f"Training set: {X_train.shape[0]} wines   |   Test set: {X_test.shape[0]} wines")
print("Class share  full : ", np.round(np.bincount(y) / len(y), 3))
print("Class share  train: ", np.round(np.bincount(y_train) / len(y_train), 3))
print("Class share  test : ", np.round(np.bincount(y_test) / len(y_test), 3))

**What the code does.** Splits the data 75 % / 25 % into training and test sets. `stratify=y` keeps each variety's share identical in both parts, and `random_state=42` makes the split reproducible.

**Why it matters.** The test set plays the role of "new, unseen wines". It must be untouched by scaling, PCA and hyper-parameter tuning, otherwise the final accuracy would be over-optimistic.

**Interpretation.** 133 training wines and 45 test wines; class proportions are virtually identical in all three sets (≈ 0.33 / 0.40 / 0.27).

**Decision.** All fitting and tuning from now on uses **only the 133 training wines**; the 45 test wines are used once, at the end, for evaluation.

## 2.3 Apply PCA to reduce dimensionality while retaining 95% variance

In [ ]:
# Explore on training data only (scaler + PCA fitted on X_train)
scaler_eda = StandardScaler().fit(X_train)
Z_train    = scaler_eda.transform(X_train)

pca_full = PCA(random_state=RANDOM_STATE).fit(Z_train)
cum_var  = np.cumsum(pca_full.explained_variance_ratio_)
n_95     = int(np.argmax(cum_var >= 0.95) + 1)

# The exact setting used in the final pipeline: keep the smallest number of PCs reaching 95 %
pca_95 = PCA(n_components=0.95, random_state=RANDOM_STATE).fit(Z_train)
print(f"Original features            : {X.shape[1]}")
print(f"Components for >= 95 % var.  : {pca_95.n_components_}   (check: {n_95})")
print(f"Variance actually retained   : {pca_95.explained_variance_ratio_.sum():.2%}")
display(pd.DataFrame({"explained variance": pca_full.explained_variance_ratio_,
                      "cumulative": cum_var},
                     index=[f"PC{i+1}" for i in range(len(cum_var))]).head(n_95 + 1).T)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
ax[0].bar(range(1, len(cum_var) + 1), pca_full.explained_variance_ratio_, color="steelblue", label="Individual")
ax[0].plot(range(1, len(cum_var) + 1), cum_var, "o-", color="darkorange", label="Cumulative")
ax[0].axhline(0.95, color="red", ls="--", label="95 % target"); ax[0].axvline(n_95, color="grey", ls=":")
ax[0].set(xlabel="Principal component", ylabel="Explained variance ratio", title="Scree plot + cumulative variance"); ax[0].legend()

P2 = PCA(2, random_state=RANDOM_STATE).fit_transform(Z_train)
sns.scatterplot(x=P2[:, 0], y=P2[:, 1], hue=[class_names[i] for i in y_train], palette="Set1", s=60, ax=ax[1])
ax[1].set(xlabel="PC1", ylabel="PC2", title="Training wines projected on the first two PCs"); ax[1].legend(loc="lower left")
plt.tight_layout(); plt.show()

**What the code does.** Standardizes the training data, fits a full PCA, and measures how much variance each principal component (PC) explains. `PCA(n_components=0.95)` automatically keeps the smallest number of PCs whose cumulative explained variance reaches 95 %. The scree plot (left) shows the individual and cumulative variance; the scatter plot (right) shows the wines on the first two PCs.

**Why it matters.** PCA rotates the 13 correlated chemical measurements into uncorrelated components ordered by information content. Dropping the weakest ones removes noise and redundancy, which helps distance-based classifiers such as k-NN.

**Interpretation.**
* **10 of 13 components** are needed to reach 95 %; they retain **96.4 %** of the variance.
* PC1 alone explains 35.7 % and PC2 19.2 %; together 55 %. The remaining variance is spread across many small components, so the compression from 13 → 10 is real but modest – the wine features are only moderately correlated.
* In the 2-D projection, the three varieties form **three visibly distinct groups**, with only a limited overlap between Class 1 and its neighbours. This signals that the chemistry carries enough information for a distance-based classifier to work well.

**Decision.** Use `PCA(n_components=0.95)` as the second stage of the pipeline (10 components in this training set), a sensible balance between compression and information retained.

## 2.4 Use GridSearchCV to find the optimal k-value and distance metric

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca",    PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ("knn",    KNeighborsClassifier()),
])

param_grid = {
    "knn__n_neighbors": list(range(1, 26)),                                # k
    "knn__metric":      ["euclidean", "manhattan", "chebyshev", "cosine"], # distance metric
    "knn__weights":     ["uniform", "distance"],                           # extra: do closer neighbours count more?
}
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(pipe, param_grid, cv=cv5, scoring="accuracy", n_jobs=-1, refit=True)
grid.fit(X_train, y_train)

print(f"Combinations tested : {len(grid.cv_results_['params'])}  x 5 folds")
print("Best parameters     :", grid.best_params_)
print(f"Best CV accuracy    : {grid.best_score_:.4f}")

res = (pd.DataFrame(grid.cv_results_)
         .loc[:, ["param_knn__n_neighbors", "param_knn__metric", "param_knn__weights", "mean_test_score", "std_test_score", "rank_test_score"]]
         .rename(columns=lambda c: c.replace("param_knn__", "")).sort_values("rank_test_score"))
print("\nTop 10 configurations:")
display(res.head(10).reset_index(drop=True))

# Heat-map: best CV accuracy for each (k, metric)
heat = res.pivot_table(index="metric", columns="n_neighbors", values="mean_test_score", aggfunc="max")
plt.figure(figsize=(15, 3.2))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="YlGnBu", annot_kws={"size": 7}, cbar_kws={"label": "CV accuracy"})
plt.title("Cross-validated accuracy by k and distance metric (best of uniform / distance weighting)")
plt.xlabel("k (n_neighbors)"); plt.tight_layout(); plt.show()

**What the code does.** Builds a `Pipeline` (StandardScaler → PCA 95 % → k-NN) and uses `GridSearchCV` with stratified 5-fold cross-validation on the training set to test **200 hyper-parameter combinations**: k from 1 to 25, four distance metrics (Euclidean, Manhattan, Chebyshev, cosine) and two voting schemes (`uniform` = every neighbour counts equally, `distance` = closer neighbours count more). Scaler and PCA are re-fitted inside every fold, so there is no leakage. The table lists the top 10 configurations and the heat-map shows the accuracy for every (k, metric) pair.

**Why it matters.** k-NN's behaviour depends heavily on k (too small = noisy, too large = blurry) and on how "closeness" is measured. Systematic tuning is more reliable than guessing.

**Interpretation.**
* **Best configuration:** `metric = cosine`, `k = 3`, `weights = distance`, with **97.8 % cross-validated accuracy**. It is tied for first place with k = 5 (same metric and weighting).
* Many other settings are practically as good (Manhattan k = 12 → 97.7 %; cosine k = 7–8 → 97.7 %; Chebyshev k = 3 → 97.0 %). The standard deviation across folds is ≈ 3 percentage points, which is larger than the gaps between the top candidates – so the top models are **statistically indistinguishable**.
* The heat-map shows a very flat landscape: **every one of the 100 (k, metric) cells lies between about 94 % and 98 %**. Euclidean distance is remarkably stable (≈ 96–97 % for every k up to 25), whereas cosine is the best at small k (up to 98 %) but slips to 94–95 % for k ≥ 16, and Chebyshev and Manhattan are slightly lower on average.

**Decision.** We adopt the grid-search winner (`cosine`, `k = 3`, `distance` weighting), as the lab requires. Because the plateau is wide, the result is robust: no tuning choice is critical, and the final accuracy does not rest on a "lucky" parameter. (Differences between the ranked candidates should not be over-interpreted.)

## 2.5 Train the k-NN classifier using the best parameters

In [ ]:
best = {k.replace("knn__", ""): v for k, v in grid.best_params_.items()}
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("pca",    PCA(n_components=0.95, random_state=RANDOM_STATE)),
    ("knn",    KNeighborsClassifier(**best)),
]).fit(X_train, y_train)

print("Final k-NN settings        :", best)
print("PCA components retained    :", final_model.named_steps["pca"].n_components_, "of", X.shape[1])
print(f"Variance retained          : {final_model.named_steps['pca'].explained_variance_ratio_.sum():.2%}")
print(f"Cross-validated accuracy   : {grid.best_score_:.4f}   (training accuracy is not reported: with distance weighting every training point is its own nearest neighbour, so it is always 100 %)")

**What the code does.** Re-creates the pipeline with the winning hyper-parameters and trains it on the entire training set (scaler → PCA → k-NN).

**Why it matters.** This is the final "production" model that will be evaluated on the untouched test wines.

**Interpretation.** The final model uses k = 3 neighbours with cosine distance and distance weighting, working on **10 principal components that retain 96.4 % of the variance**. Its cross-validated accuracy is 97.8 %. Training accuracy is deliberately not reported: with distance weighting each training point is its own nearest neighbour (distance 0), so it would trivially read 100 % and say nothing about generalisation.

**Decision.** Proceed to evaluate this model on the 45 held-out test wines.

## 2.6 Evaluate the model with a classification report and accuracy score

In [ ]:
y_pred = final_model.predict(X_test)
print(f"TEST ACCURACY: {accuracy_score(y_test, y_pred):.4f}   ({(y_pred == y_test).sum()} of {len(y_test)} wines correct)\n")
print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=class_names, cmap="Blues", ax=ax[0], colorbar=False)
ax[0].set_title("Confusion matrix (counts)")
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=class_names, cmap="Blues", normalize="true",
                                        values_format=".0%", ax=ax[1], colorbar=False)
ax[1].set_title("Confusion matrix (% of true class)")
plt.tight_layout(); plt.show()

print("Misclassified test wines:")
display(pd.DataFrame({"true": [class_names[i] for i in y_test], "predicted": [class_names[i] for i in y_pred]},
                     index=X_test.index)[y_test != y_pred])

**What the code does.** Predicts the varieties of the 45 test wines and reports the accuracy score, a classification report (precision, recall, F1 per class) and a confusion matrix (counts and percentages). It also lists the wines that were misclassified.

**Why it matters.** Accuracy alone hides *which* varieties are confused. Precision answers *"when the model says Class X, how often is it right?"*; recall answers *"of all true Class X wines, how many did we find?"*.

**Interpretation.**
* **Test accuracy = 93.3 %** (42 of 45 wines correct).
* **Class 0:** recall 100 % (all 15 found), precision 93.8 %. **Class 2:** recall 100 % (all 12 found), precision 85.7 %. **Class 1:** precision 100 % but recall 83.3 % (15 of 18 found).
* The **three errors are all true Class 1 wines**: one was labelled Class 0 and two Class 2. The model never confuses Class 0 with Class 2, which are chemically the most different. The macro-average F1 is 0.933, so the performance is balanced across the classes.
* With only 45 test wines, **one wine = 2.2 percentage points**, so this figure is subject to sampling noise.

**Decision.** The model is accurate enough for a prototype: it correctly identifies roughly 14 out of 15 wines. For quality control we recommend that a wine predicted as Class 0 or 2 whose chemistry sits near the Class 1 boundary be flagged for manual verification, because Class 1 is the variety that is most often mis-identified. The next section checks how trustworthy the 93 % figure really is.

## 2.7 Robustness checks: baselines and nested cross-validation *(additional step)*
A single 45-wine test set is small, so we also (i) compare against simpler pipelines to prove that scaling, PCA and tuning each add value, and (ii) run **nested cross-validation** that repeats the *whole* tuning procedure inside each outer fold, giving an honest estimate of how the approach generalises.

In [ ]:
candidates = {
    "1. Raw features, default k-NN (k=5)":            Pipeline([("knn", KNeighborsClassifier())]),
    "2. Standardized, default k-NN":                  Pipeline([("s", StandardScaler()), ("knn", KNeighborsClassifier())]),
    "3. Standardized + PCA(95%), default k-NN":       Pipeline([("s", StandardScaler()), ("p", PCA(0.95, random_state=RANDOM_STATE)), ("knn", KNeighborsClassifier())]),
    "4. Standardized + PCA(95%) + tuned k-NN (final)": final_model,
}
rows = []
for name, model in candidates.items():
    m = model if name.startswith("4") else model.fit(X_train, y_train)
    cv = cross_val_score(m if name.startswith("4") else model, X_train, y_train, cv=cv5)  # same folds for all
    rows.append({"pipeline": name, "5-fold CV acc (train)": cv.mean(), "test accuracy": m.score(X_test, y_test)})
baseline = pd.DataFrame(rows).set_index("pipeline")
display(baseline)

# Nested CV: outer 5 folds on ALL data, inner GridSearchCV re-tunes in every outer fold
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
nested = cross_val_score(GridSearchCV(pipe, param_grid, cv=cv5, n_jobs=-1), X, y, cv=outer, n_jobs=1)
print("Nested CV accuracy per outer fold:", np.round(nested, 3))
print(f"Nested CV accuracy: {nested.mean():.3f} ± {nested.std():.3f}")

baseline["test accuracy"].plot.barh(color=["#bbb", "#9ecae1", "#6baed6", "#08519c"], figsize=(9, 3))
plt.xlim(0.5, 1.0); plt.xlabel("Test accuracy"); plt.title("Each preprocessing / tuning step improves k-NN"); plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

**What the code does.** (i) Compares four pipelines of increasing sophistication on the same folds and test set; (ii) runs **nested cross-validation**: an outer 5-fold loop on *all* 178 wines in which the entire grid search is repeated inside every training fold, which gives a nearly unbiased estimate of the whole "tune-then-predict" procedure.

**Why it matters.** A single 45-wine test set is a small sample, and test accuracy must not be used to pick a model (that would turn the test set into a second tuning set). These checks quantify how much each step contributes and how reliable the accuracy is.

**Interpretation.**
* **Scaling is the biggest single gain:** raw k-NN scores only 66 % (CV) / 77.8 % (test), whereas standardized k-NN reaches 95.5 % / 93.3 %. Features such as proline were dominating the distance calculation.
* **PCA adds a little** (CV 96.3 %, test 95.6 %): it removes noise without losing information.
* **Tuning gives the best CV score** (97.8 %). On the 45-wine test set, the tuned model (93.3 %) is one wine behind the default-k PCA model (95.6 %); a one-wine difference is well within sampling noise, and we do not choose between models on the test set.
* **Nested CV accuracy = 95.5 % ± 1.4 %** across the five outer folds (94.3–97.2 %). This is the most dependable estimate of real-world performance.

**Decision.** The pipeline *Standardize → PCA (95 %) → tuned k-NN* is retained. We report **≈ 95 % expected accuracy** (nested CV) as the headline figure for the client, with the 93.3 % held-out score as a consistent, slightly conservative confirmation.

---
# Step 3 — Recommendation System Using the Chickwts Dataset
*Goal: given a feed a farmer already uses, recommend other feeds likely to produce similar chick growth. Chicken weight is used as the performance proxy.*

## 3.1 Standardize the weight feature for uniform scaling

In [ ]:
chick_rs = chick.copy()
scaler_w = StandardScaler().fit(chick_rs[["weight"]])
chick_rs["weight_z"] = scaler_w.transform(chick_rs[["weight"]]).ravel()

print(f"Raw weight      : mean = {chick_rs.weight.mean():.1f} g, std = {chick_rs.weight.std(ddof=0):.1f} g")
print(f"Standardized    : mean = {chick_rs.weight_z.mean():.3f},   std = {chick_rs.weight_z.std(ddof=0):.3f}")
display(chick_rs.sample(5, random_state=RANDOM_STATE))

**What the code does.** Standardizes `weight` (grams) to z-scores: z = (weight − 261.3) / 77.5. A random sample of rows shows raw and standardized values side by side.

**Why it matters.** The lab requires uniform scaling before PCA. It puts weight on a unit-free scale (mean 0, spread 1) and makes the method reusable if further performance variables (e.g. feed conversion, mortality) are added later.

**Interpretation.** The average chick weighs 261 g (SD ≈ 77 g). A horsebean chick of 179 g becomes z = −1.06 (about one SD below average), a soybean chick of 243 g becomes −0.24 (just below average).

**Decision.** Use `weight_z` as the input to PCA.

## 3.2 Apply PCA to reduce data to one principal component

In [ ]:
pca_w = PCA(n_components=1, random_state=RANDOM_STATE)
pc1   = pca_w.fit_transform(chick_rs[["weight_z"]]).ravel()

# PCA's sign is arbitrary; orient it so that a HIGHER score always means a HEAVIER chick
if pca_w.components_[0, 0] < 0:
    pc1 = -pc1
chick_rs["pc1"] = pc1

print("Explained variance ratio of PC1 :", pca_w.explained_variance_ratio_.round(4))
print("Correlation(PC1, raw weight)    : %.4f" % np.corrcoef(chick_rs.pc1, chick_rs.weight)[0, 1])

# Why cosine similarity cannot be applied directly to single numbers:
a, b = np.array([[0.2]]), np.array([[1.9]])
print("\nCosine similarity between two 1-D values 0.2 and 1.9 :", cosine_similarity(a, b)[0, 0])
print("Cosine similarity between 1-D values 0.2 and -1.9     :", cosine_similarity(a, -b)[0, 0])

**What the code does.** Reduces the standardized weight to one principal component and orients its sign so that a higher score always means a heavier chick. It then tests what cosine similarity does with single numbers.

**Why it matters.** The lab specifies "one principal component". It is important to understand what that does here.

**Interpretation.**
* With **only one input feature**, PCA cannot compress anything: PC1 explains 100 % of the variance and is perfectly correlated (r = 1.00) with the raw weight. It is simply the standardized weight (the sign is arbitrary in PCA, which is why we orient it).
* The second part exposes a **mathematical problem**: the cosine similarity of two one-dimensional numbers is always **+1 or −1**, since cosine measures only the angle between vectors, and on a line there are just two directions. Comparing individual chicks, or even two feed averages, would give only "same sign" vs. "different sign", which is useless for ranking feeds.

**Decision.** Cosine similarity needs a *vector* for each feed, so rather than dropping the method we describe each feed by a profile of its PC1 distribution (next sub-section). This keeps all the required elements (standardization → PC1 → cosine similarity) while making the similarity scores meaningful.

## 3.3 Build one profile vector per feed *(additional step required to make cosine similarity meaningful)*

In [ ]:
# Summarise each feed's distribution of PC1 scores with five robust statistics (a 5-number summary)
def five_number(s):
    return pd.Series({"min": s.min(), "q25": s.quantile(.25), "median": s.median(),
                      "q75": s.quantile(.75), "max": s.max()})

feed_profile = chick_rs.groupby("feed")["pc1"].apply(five_number).unstack()
feed_profile = feed_profile.loc[chick.groupby("feed")["weight"].mean().sort_values().index]   # order light -> heavy
feed_stats   = chick.groupby("feed")["weight"].agg(n="count", mean_weight_g="mean", sd_g="std").round(1)

display(feed_profile)

**What the code does.** For every feed, summarises the PC1 scores of its chicks by five numbers: minimum, 25th percentile, median, 75th percentile and maximum. Each feed therefore becomes a 5-dimensional vector. Feeds are listed from lightest to heaviest.

**Why it matters.** The five-number summary captures both the *location* (how heavy the chicks grow) and the *spread/shape* (consistency) of a feed's outcome, and it is robust to outliers. It also gives each feed as many comparable data points as the other feeds, despite unequal group sizes (10–14 chicks).

**Interpretation.** The profiles form a clear gradient. Horsebean sits entirely below the average (all values negative, median −1.42), linseed and soybean straddle slightly below average, meatmeal is centred near the average (median 0.02), and casein and sunflower are almost wholly above average (medians 1.04 and 0.86).

**Decision.** These six vectors are the input to the cosine similarity computation.

## 3.4 Compute cosine similarity between different feed types

In [ ]:
cos_sim = pd.DataFrame(cosine_similarity(feed_profile), index=feed_profile.index, columns=feed_profile.index)
display(cos_sim.round(3))

plt.figure(figsize=(7.5, 6))
sns.heatmap(cos_sim, annot=True, fmt=".2f", cmap="RdYlGn", vmin=-1, vmax=1, square=True, cbar_kws={"label": "cosine similarity"})
plt.title("Feed-to-feed cosine similarity (feeds ordered light → heavy)")
plt.tight_layout(); plt.show()

**What the code does.** Computes the cosine similarity between every pair of feed profiles (cosine of the angle between the vectors: **+1** = same pattern, **0** = unrelated, **−1** = opposite) and shows the 6 × 6 matrix as a heat-map. Rows and columns run from the lightest to the heaviest feed.

**Why it matters.** This matrix is the engine of the recommender: the more similar the profiles, the more comparable the growth results a farmer can expect.

**Interpretation.**
* **Casein ↔ sunflower: 0.965** and **linseed ↔ soybean: 0.957** are the two most similar pairs; soybean ↔ meatmeal (0.873) and linseed ↔ horsebean (0.822) follow.
* **Horsebean is the outlier:** its similarity is negative to casein (−0.37) and to sunflower (−0.42). Its profile points in the opposite direction from the high-growth feeds.
* The heat-map is organised in a visible gradient: similarity decays smoothly as the feeds move apart in the light-to-heavy order, which is exactly the behaviour a sensible similarity measure should show.

**Decision.** Use this matrix to rank recommendations, and apply thresholds to describe their quality: **strong ≥ 0.90**, **moderate 0.70–0.90**, **weak < 0.70**.

## 3.5 Recommend similar feeds based on similarity scores

In [ ]:
def strength(s):
    return "strong" if s >= 0.90 else "moderate" if s >= 0.70 else "weak"

def recommend_similar_feeds(feed, top_n=3, min_similarity=0.70):
    """Return the top_n feeds most similar to `feed`, flagged as strong / moderate / weak matches."""
    feed = feed.lower().strip()
    if feed not in cos_sim.index:
        raise ValueError(f"Unknown feed '{feed}'. Choose from {list(cos_sim.index)}")
    s = cos_sim[feed].drop(feed).sort_values(ascending=False).head(top_n)
    out = pd.DataFrame({
        "recommended_feed":  s.index,
        "cosine_similarity": s.values,
        "match":             [strength(v) for v in s.values],
        "mean_weight_g":     feed_stats.loc[s.index, "mean_weight_g"].values,
        "weight_gap_g":      (feed_stats.loc[s.index, "mean_weight_g"] - feed_stats.loc[feed, "mean_weight_g"]).abs().values,
    })
    if (out["cosine_similarity"] < min_similarity).all():
        print(f"⚠ No close substitute exists for '{feed}' (best similarity below {min_similarity}).")
    return out

for f in cos_sim.index:
    print(f"\n▶ Farmer currently uses  {f.upper()}  (mean weight {feed_stats.loc[f, 'mean_weight_g']:.0f} g)")
    display(recommend_similar_feeds(f, top_n=2).round(3).style.hide(axis="index"))

**What the code does.** Defines `recommend_similar_feeds(feed, top_n)`, a reusable function that returns the most similar feeds with their similarity score, a strong/moderate/weak label, the feed's average weight and the average-weight gap to the current feed. It then prints the top-2 recommendations for every feed. It also warns when no feed reaches the moderate threshold.

**Why it matters.** This is the actual deliverable for the agricultural supply company: for a farmer's current feed, it suggests comparable alternatives (e.g. when a feed is out of stock or too expensive) and says how much confidence the recommendation deserves.

**Interpretation (top recommendations; see the tables above).**

| Farmer uses | 1st recommendation | 2nd recommendation |
|---|---|---|
| Horsebean (160 g) | Linseed – 0.82, moderate (≈ 59 g heavier) | Soybean – 0.63, weak |
| Linseed (219 g) | Soybean – 0.96, strong | Horsebean – 0.82, moderate |
| Soybean (246 g) | Linseed – 0.96, strong | Meatmeal – 0.87, moderate |
| Meatmeal (277 g) | Soybean – 0.87, moderate | Casein – 0.81, moderate |
| Casein (324 g) | Sunflower – 0.97, strong | Meatmeal – 0.81, moderate |
| Sunflower (329 g) | Casein – 0.97, strong | Meatmeal – 0.79, moderate |

* **Casein and sunflower** are practically interchangeable, as are **linseed and soybean**.
* **Meatmeal** is a "bridge" feed, moderately similar to both the heavier and the lighter groups.
* **Horsebean has no close substitute.** Its best match is only moderate, and its second is weak. Moreover, similarity here means "similar pattern", so the nearest feed (linseed) still gives chicks that are ~59 g heavier on average. For a horsebean user this is a performance *upgrade*, not a like-for-like swap.

**Decision.** The engine is ready as a prototype. We recommend that the client only advertise "strong" matches as direct substitutes, present "moderate" ones as "comparable alternatives" and treat any result for horsebean as an upgrade suggestion.

## 3.6 Validate the recommendations *(additional step)*

In [ ]:
# Independent sanity check: do the recommendations agree with the simple fact of "closest average weight"?
mean_w   = feed_stats["mean_weight_g"]
gap      = pd.DataFrame({a: (mean_w - mean_w[a]).abs() for a in mean_w.index})
gap      = gap.loc[cos_sim.index, cos_sim.columns]          # align rows/columns with the similarity matrix
rows = []
for f in cos_sim.index:
    rows.append({"feed": f,
                 "top cosine recommendation": cos_sim[f].drop(f).idxmax(),
                 "closest by mean weight":    gap[f].drop(f).idxmin()})
agree = pd.DataFrame(rows).set_index("feed"); agree["agree?"] = agree.iloc[:, 0] == agree.iloc[:, 1]
display(agree)

iu = np.triu_indices(len(cos_sim), k=1)
rho, p = spearmanr(cos_sim.values[iu], -gap.values[iu])
print(f"Spearman correlation between cosine similarity and (−) mean-weight gap, across all {len(iu[0])} feed pairs: ρ = {rho:.2f}  (p = {p:.4f})")

order = feed_stats.sort_values("mean_weight_g").index
plt.figure(figsize=(9, 4))
sns.boxplot(data=chick, x="weight", y="feed", order=order, color="#9ecae1")
sns.stripplot(data=chick, x="weight", y="feed", order=order, color="k", size=3, alpha=.6)
plt.title("Chick weight by feed — feeds with overlapping boxes are natural substitutes"); plt.xlabel("Weight (g)")
plt.tight_layout(); plt.show()

**What the code does.** Tests the recommender against a deliberately simple, independent benchmark: *"which feed has the closest average weight?"*. It compares the top recommendation with that benchmark for all six feeds, calculates the Spearman rank correlation between cosine similarity and weight proximity across all 15 feed pairs, and draws the raw chick weights per feed.

**Why it matters.** A recommender with no ground truth must at least agree with common sense, and the chart lets stakeholders see the raw evidence behind the scores.

**Interpretation.**
* **All six top recommendations coincide** with the feed that has the closest average weight.
* Across all 15 pairs, cosine similarity and weight proximity are almost perfectly rank-aligned (**Spearman ρ = 0.96**, p < 0.001).
* The box-plots confirm the picture: casein and sunflower boxes overlap almost entirely; linseed/soybean overlap strongly; horsebean overlaps only with the lower tails of linseed and soybean.
* **Limitations:** each feed has only 10–14 chicks, and weight is the *only* performance measure; neither price, nutrient content nor feed conversion is available.

**Decision.** The PCA + cosine approach is validated as a faithful similarity ranking. Because the data are small and one-dimensional in substance, treat the scores as *guidance*; before a real deployment, add richer features (cost, protein content, feed-conversion ratio, mortality) to the profile vectors so that the recommender can distinguish feeds with the same growth but different economics.

---
# Step 4 — Clustering (K-Means and GMM) on the USArrests Dataset
*Goal: discover natural groups of US states with similar crime profiles, so that policy research can target regions with similar needs.*

## 4.1 Standardize the dataset for consistency

In [ ]:
print("Raw standard deviations (the features live on very different scales):")
display(arrests.std().to_frame("std (raw)").T)

arrests_scaled = pd.DataFrame(StandardScaler().fit_transform(arrests), index=arrests.index, columns=arrests.columns)
print("After standardization:")
display(arrests_scaled.describe().loc[["mean", "std"]].round(3))

**What the code does.** Shows the raw spreads of the four variables and re-applies standardization (re-using the approach from Step 1.3) to produce `arrests_scaled`.

**Why it matters.** K-Means and PCA are distance/variance based. Raw `Assault` values (SD ≈ 83) would overwhelm `Murder` (SD ≈ 4.4) simply because the numbers are larger.

**Interpretation.** Raw standard deviations differ by a factor of almost 20 (4.4 for Murder vs. 83.3 for Assault). After scaling, every column has mean 0 and standard deviation 1 (displayed as 1.01 because `describe()` uses the *n−1* convention).

**Decision.** Use `arrests_scaled` for feature selection, PCA and clustering, so that each crime variable is treated on equal footing.

## 4.2 Select the top 3 relevant features for clustering
Clustering is unsupervised (there is no target to correlate against), so we judge each feature by three complementary, unsupervised criteria: **(a)** redundancy with the other features, **(b)** contribution to the main PCA directions, and **(c)** how well the data cluster when each feature is left out.

In [ ]:
# (a) Redundancy: how strongly does each feature overlap with the others?
corr = arrests.corr()
redundancy = (corr.abs().sum() - 1) / (len(corr) - 1)           # mean |r| with the other three features
display(corr.round(2))
print("Mean absolute correlation with the other features:"); display(redundancy.round(2).to_frame("mean |r|").T)

# (b) Contribution to the first two principal components, weighted by the variance they explain
pca4 = PCA(random_state=RANDOM_STATE).fit(arrests_scaled)
print("\nVariance explained by PC1..PC4:", pca4.explained_variance_ratio_.round(3))
display(pd.DataFrame(pca4.components_[:2], index=["PC1", "PC2"], columns=arrests.columns).round(2))
importance = (pca4.components_[:2] ** 2 * pca4.explained_variance_ratio_[:2, None]).sum(axis=0)
print("PCA-loading importance (PC1+PC2, variance-weighted):"); display(pd.Series(importance, index=arrests.columns).round(3).to_frame("importance").T)

**What the code does.** Examines the features through two unsupervised lenses: **(a)** the correlation matrix and each feature's average overlap (mean |r|) with the other three; **(b)** a PCA of all four variables, showing the loadings of PC1/PC2 and a variance-weighted "importance" score per feature.

**Why it matters.** The lab asks us to keep the *top 3* of 4 features. Clustering has no target variable, so we must define "relevant" using the structure of the data itself.

**Interpretation.**
* **Redundancy:** `Murder` and `Assault` are strongly correlated (r = 0.80); together with `Rape` they form a tight "violent-crime" family (r = 0.56–0.80). **`UrbanPop` is the odd one out** (mean |r| = 0.25 vs. 0.48–0.58 for the others; r = 0.07 with Murder).
* **PCA:** PC1 (62 % of variance) gives roughly equal weights to Murder, Assault and Rape (0.54–0.58) and very little to UrbanPop (0.28): it is a general "violent-crime level" axis. PC2 (25 %) is dominated by UrbanPop (0.87).
* **Loading importance** is almost flat (0.19–0.24), so on its own it cannot rank features. It even scores UrbanPop highest, simply because UrbanPop owns PC2 – which says "UrbanPop varies independently of crime", not "UrbanPop describes crime patterns".

**Decision.** The two lenses disagree, so they cannot settle the question alone. We need a test that measures what we actually care about – how well the data cluster – and we run it next.

In [ ]:
# (c) Wrapper-style test: leave one feature out, keep the other three, reduce to 2 PCs, and
#     measure cluster quality (silhouette, averaged over k = 2..5 so the result does not hinge on one k).
rows = []
for subset in combinations(arrests.columns, 3):
    Ps = PCA(2, random_state=RANDOM_STATE).fit_transform(arrests_scaled[list(subset)])
    sil = {k: silhouette_score(Ps, KMeans(k, n_init=20, random_state=RANDOM_STATE).fit_predict(Ps)) for k in range(2, 6)}
    rows.append({"left out": (set(arrests.columns) - set(subset)).pop(), "kept features": ", ".join(subset),
                 **{f"k={k}": v for k, v in sil.items()}, "mean silhouette": np.mean(list(sil.values()))})
subset_scores = pd.DataFrame(rows).sort_values("mean silhouette", ascending=False).reset_index(drop=True)
display(subset_scores)

SELECTED = subset_scores.loc[0, "kept features"].split(", ")
print("\nSELECTED FEATURES:", SELECTED)

subset_scores.set_index("left out")["mean silhouette"].sort_values().plot.barh(color="steelblue", figsize=(7, 2.6))
plt.xlabel("Mean silhouette (k = 2..5)"); plt.title("Cluster quality when a feature is left out"); plt.tight_layout(); plt.show()

**What the code does.** Leaves out each feature in turn, keeps the other three, reduces them to 2 PCs and runs K-Means for k = 2…5, measuring the **silhouette score** (how well-separated and compact the clusters are; higher is better, max 1). Averaging over k = 2…5 prevents the answer from hinging on one value of k. The feature set with the best average silhouette is selected automatically.

**Why it matters.** This is a direct, wrapper-style test of "which three features give the most meaningful clusters?".

**Interpretation.**
* Dropping **UrbanPop** gives the best clustering quality (mean silhouette **0.479**), clearly ahead of dropping Rape (0.413), Assault (0.392) or Murder (0.388). The ranking holds for every individual k (e.g. 0.568 at k = 2 vs. ≤ 0.432 for the alternatives).
* In other words, `UrbanPop` *dilutes* the crime-based group structure, while Murder, Assault and Rape each contribute useful, partly complementary information.

**Decision.** **Selected features: Murder, Assault and Rape.** This also makes domain sense: the aim is a *crime pattern* analysis, and these are the three crime measures, whereas the urban share of the population is a demographic context variable. (We will still use `UrbanPop` afterwards to describe the clusters, and a later table shows it barely differs between them – a further justification for dropping it.)

## 4.3 Apply PCA to reduce the dataset to 2 principal components

In [ ]:
Z3 = arrests_scaled[SELECTED]
pca2 = PCA(n_components=2, random_state=RANDOM_STATE).fit(Z3)
P = pca2.transform(Z3)

# Orient components so that PC1 = "more crime" and PC2 = "more rape relative to murder/assault"
for i, ref in [(0, None), (1, "Rape")]:
    sign = np.sign(pca2.components_[i].sum()) if ref is None else np.sign(pca2.components_[i][SELECTED.index(ref)])
    pca2.components_[i] *= sign; P[:, i] *= sign

loadings = pd.DataFrame(pca2.components_.T, index=SELECTED, columns=["PC1", "PC2"])
print("Explained variance ratio:", pca2.explained_variance_ratio_.round(3), "| total = %.1f %%" % (100 * pca2.explained_variance_ratio_.sum()))
display(loadings.round(2))

P_df = pd.DataFrame(P, index=arrests.index, columns=["PC1", "PC2"])
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.scatter(P_df.PC1, P_df.PC2, s=35, color="steelblue")
for st, r in P_df.iterrows(): ax.annotate(st, (r.PC1, r.PC2), fontsize=7, alpha=.8, xytext=(3, 3), textcoords="offset points")
ax.axhline(0, color="grey", lw=.6); ax.axvline(0, color="grey", lw=.6)
ax.set(xlabel=f"PC1 – overall violent-crime level ({pca2.explained_variance_ratio_[0]:.0%})",
       ylabel=f"PC2 – rape vs. murder/assault ({pca2.explained_variance_ratio_[1]:.0%})", title="US states in the 2-D PCA space")
plt.tight_layout(); plt.show()

**What the code does.** Fits a 2-component PCA on the three selected, standardized features, orients the axes for readability (positive PC1 = more crime; positive PC2 = more rape relative to murder/assault), prints the explained variance and loadings, and plots the 50 states in the new 2-D space.

**Why it matters.** Two dimensions can be plotted and are enough to capture almost all the information, which simplifies clustering and makes the results explainable to non-technical audiences.

**Interpretation.**
* The two components retain **93.9 %** of the variance of the three selected features (PC1 = 78.6 %, PC2 = 15.3 %), so very little information is lost.
* **PC1 – overall violent-crime level:** all three loadings are strongly positive and nearly equal (0.54–0.61). A state far to the right has high murder, assault *and* rape rates.
* **PC2 – type of crime:** Rape loads +0.82 while Murder loads −0.53 (Assault −0.21), so it contrasts "rape-heavy" states (top) with "murder/assault-heavy" states (bottom).
* The scatter plot shows a continuous cloud with a dense low-crime group on the left and a more dispersed high-crime group on the right.

**Decision.** Cluster in this 2-D PCA space (`P`). Since PC1 carries most of the variance, we expect the main split to be on overall crime level.

## 4.4 Determine the optimal number of clusters — K-Means elbow method (inertia)

In [ ]:
K_RANGE = range(1, 11)
inertia = [KMeans(k, n_init=20, random_state=RANDOM_STATE).fit(P).inertia_ for k in K_RANGE]

# Objective elbow: the point furthest from the straight line joining the first and last inertia values
x = np.array(list(K_RANGE)); yv = np.array(inertia)
v = np.array([x[-1] - x[0], yv[-1] - yv[0]])
dist = np.abs(v[0] * (yv - yv[0]) - v[1] * (x - x[0])) / np.linalg.norm(v)
k_elbow = int(x[dist.argmax()])

# Supporting evidence: silhouette score (k >= 2)
sil_km = {k: silhouette_score(P, KMeans(k, n_init=20, random_state=RANDOM_STATE).fit_predict(P)) for k in range(2, 11)}
k_sil = max(sil_km, key=sil_km.get)

display(pd.DataFrame({"inertia": inertia, "drop vs previous k": [np.nan] + list(-np.diff(inertia)),
                      "silhouette": [np.nan] + list(sil_km.values())}, index=pd.Index(K_RANGE, name="k")).round(2).T)
print(f"Elbow (max distance to chord) -> k = {k_elbow}      Best silhouette -> k = {k_sil}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.3))
ax[0].plot(K_RANGE, inertia, "o-"); ax[0].axvline(k_elbow, color="red", ls="--", label=f"elbow k = {k_elbow}")
ax[0].set(xlabel="Number of clusters k", ylabel="Inertia (within-cluster SSE)", title="K-Means elbow method"); ax[0].legend()
ax[1].plot(list(sil_km), list(sil_km.values()), "s-", color="seagreen"); ax[1].axvline(k_sil, color="red", ls="--")
ax[1].set(xlabel="Number of clusters k", ylabel="Silhouette score", title="Supporting evidence: silhouette")
plt.tight_layout(); plt.show()

**What the code does.** Runs K-Means for k = 1…10 and records the **inertia** (total squared distance of the states to their cluster centre; lower = tighter clusters). The "elbow" is located objectively as the point furthest from the straight line joining the first and last points. The silhouette score is shown as supporting evidence.

**Why it matters.** Adding clusters always lowers inertia, so we look for the point where extra clusters stop paying off.

**Interpretation.**
* Inertia drops steeply from **140.8 (k = 1) to 47.1 (k = 2)** – a reduction of 93.7 – then only by 14.3 (k = 3) and 13.4 (k = 4), and by progressively smaller amounts afterwards. The elbow is at **k = 2**.
* The silhouette score also peaks at **k = 2 (0.57)** and drops to 0.43–0.46 for k ≥ 3.

**Decision.** The K-Means evidence favours **k = 2**. We now check whether GMM's probabilistic criterion agrees.

## 4.5 Determine the optimal number of clusters — GMM Bayesian Information Criterion (BIC)

In [ ]:
cov_types = ["full", "tied", "diag", "spherical"]
bic_table = pd.DataFrame({ct: [GaussianMixture(k, covariance_type=ct, n_init=10, random_state=RANDOM_STATE).fit(P).bic(P)
                               for k in range(1, 9)] for ct in cov_types}, index=pd.Index(range(1, 9), name="k"))
display(bic_table.round(1))

k_bic  = int(bic_table.min(axis=1).idxmin())               # k with the overall lowest BIC
ct_bic = bic_table.loc[k_bic].idxmin()                      # covariance type that achieves it
print(f"Lowest BIC = {bic_table.values.min():.1f}  ->  k = {k_bic}, covariance_type = '{ct_bic}'")

plt.figure(figsize=(8.5, 4.5))
for ct in cov_types: plt.plot(bic_table.index, bic_table[ct], "o-", label=ct)
plt.scatter([k_bic], [bic_table.loc[k_bic, ct_bic]], s=250, facecolors="none", edgecolors="red", lw=2, label="minimum BIC")
plt.xlabel("Number of components k"); plt.ylabel("BIC (lower is better)"); plt.title("GMM model selection with BIC"); plt.legend(title="covariance")
plt.tight_layout(); plt.show()

**What the code does.** Fits GMMs with 1 to 8 components and four covariance structures (`full` = free-form ellipses, `tied` = shared ellipse, `diag` = axis-aligned ellipses, `spherical` = circles) and computes the **Bayesian Information Criterion (BIC)** for each. BIC = goodness of fit penalised by model complexity; **lower is better**.

**Why it matters.** Unlike K-Means inertia, BIC explicitly penalises complexity, so it protects against inventing clusters that merely fit noise – especially important with only 50 observations. Trying several covariance shapes lets the data choose the cluster geometry.

**Interpretation.**
* The overall minimum is **BIC = 287.8 at k = 2 with `diag` covariance**.
* The second-best option (`full`, k = 2) has BIC = 293.3, and the other covariance types are higher at k = 2. Beyond k = 2, BIC generally *increases* (e.g. `full`: 293 → 304 → 317), meaning extra clusters are not justified.
* A 1-cluster model is clearly worse (303–330), so there *is* real group structure.

**Decision.** BIC independently confirms **k = 2** and selects axis-aligned (`diag`) covariance, which is a parsimonious structure that suits PCA components, because PCA components are uncorrelated by construction.

## 4.6 Decide the final number of clusters *(additional step)*

In [ ]:
votes = {"K-Means elbow": k_elbow, "Silhouette": k_sil, "GMM BIC": k_bic}
K_FINAL = Counter(votes.values()).most_common(1)[0][0]            # majority vote
GMM_COV = bic_table.loc[K_FINAL].idxmin()                          # best covariance type at that k
print("Evidence:", votes)
print(f"=> K_FINAL = {K_FINAL};  GMM covariance type at this k = '{GMM_COV}'")

**What the code does.** Collects the three pieces of evidence (K-Means elbow, silhouette and GMM BIC) and takes a majority vote for the final number of clusters, then picks the best GMM covariance type at that k.

**Why it matters.** Relying on a single, sometimes ambiguous criterion (the elbow is notoriously subjective) is risky. Agreement of independent criteria gives the choice real credibility.

**Interpretation.** All three methods give **k = 2** (unanimous), and the preferred GMM covariance at k = 2 is `diag`.

**Decision.** **Final model: 2 clusters** for both K-Means and GMM. Section 4.10 later tests how sensitive the conclusions are to this choice.

## 4.7 Apply K-Means for hard cluster assignments

In [ ]:
def order_by_pc1(labels):
    """Re-number clusters so that cluster 0 has the lowest average PC1 (= lowest crime). Returns mapping old->new."""
    order = np.argsort([P[labels == c, 0].mean() for c in range(labels.max() + 1)])
    return {old: new for new, old in enumerate(order)}

km = KMeans(n_clusters=K_FINAL, n_init=50, random_state=RANDOM_STATE).fit(P)
m  = order_by_pc1(km.labels_)
km_labels  = np.array([m[l] for l in km.labels_])
km_centers = km.cluster_centers_[np.argsort([m[i] for i in range(K_FINAL)])]

print(f"K-Means (k={K_FINAL}):  inertia = {km.inertia_:.2f},  silhouette = {silhouette_score(P, km_labels):.3f}")
print("Cluster sizes:", dict(zip(*np.unique(km_labels, return_counts=True))))
for c in range(K_FINAL):
    print(f"\nCluster {c}: ", ", ".join(arrests.index[km_labels == c]))

**What the code does.** Fits K-Means (k = 2, 50 random initialisations to avoid a poor local optimum) on the PCA scores, and relabels clusters so that cluster 0 always has the lowest average PC1 (lowest crime) – this makes the labels comparable with the GMM and stable between runs. It prints the silhouette, cluster sizes and the member states.

**Why it matters.** K-Means yields **hard assignments**: each state belongs to exactly one cluster, which is simple to communicate and act on.

**Interpretation.**
* **Cluster 0 – 30 states (lower-crime):** e.g. Vermont, North Dakota, Maine, Wisconsin, Iowa, Minnesota, Massachusetts, Connecticut, Utah, Washington, Oregon…
* **Cluster 1 – 20 states (higher-crime):** e.g. Alabama, Alaska, Arizona, California, Florida, Georgia, Illinois, Louisiana, Maryland, Michigan, Mississippi, Nevada, New Mexico, New York, North and South Carolina, Texas…
* The silhouette score of **0.568** is *moderate to good* (> 0.5 means the structure is reasonable), with a total inertia of 47.1.

**Decision.** K-Means gives a clean two-segment partition. Next, GMM tells us *how confident* each assignment is.

## 4.8 Apply GMM for probabilistic clustering

In [ ]:
gmm = GaussianMixture(n_components=K_FINAL, covariance_type=GMM_COV, n_init=20, random_state=RANDOM_STATE).fit(P)
raw_lab = gmm.predict(P); m = order_by_pc1(raw_lab)
gmm_labels = np.array([m[l] for l in raw_lab])
inv = np.argsort([m[i] for i in range(K_FINAL)])                      # new column order of the posterior matrix
gmm_proba = pd.DataFrame(gmm.predict_proba(P)[:, inv], index=arrests.index, columns=[f"P(cluster {c})" for c in range(K_FINAL)])
gmm_proba["assigned"]   = gmm_labels
gmm_proba["confidence"] = gmm_proba.iloc[:, :K_FINAL].max(axis=1)

print(f"GMM (k={K_FINAL}, covariance='{GMM_COV}'):  BIC = {gmm.bic(P):.1f},  silhouette = {silhouette_score(P, gmm_labels):.3f},  converged = {gmm.converged_}")
print("Mixing weights :", np.round(gmm.weights_[inv], 3))
print("Cluster sizes  :", dict(zip(*np.unique(gmm_labels, return_counts=True))))

UNCERTAIN = 0.90
unc = gmm_proba[gmm_proba.confidence < UNCERTAIN].sort_values("confidence")
print(f"\nStates whose cluster membership is uncertain (max posterior < {UNCERTAIN:.0%}): {len(unc)}")
display(unc.round(3))
print("Distribution of membership confidence:"); display(gmm_proba["confidence"].describe().round(3).to_frame("confidence").T)

**What the code does.** Fits a 2-component GMM with `diag` covariance (20 initialisations), relabels the clusters in the same order, and extracts the **posterior membership probabilities** – the model's confidence that each state belongs to each cluster. States with a top probability below 90 % are flagged as uncertain.

**Why it matters.** Crime profiles form a continuum, and real states can sit between the groups. A probabilistic model reports this uncertainty instead of hiding it behind a hard label.

**Interpretation.**
* The model converged; it splits the states 31 / 19 (mixing weights 62 % / 38 %).
* **47 of 50 states have ≥ 90 % membership confidence** (median ≈ 100 %, mean 98.4 %), so the two groups are well separated.
* Only three states are uncertain: **Missouri (69 % cluster 0 / 31 % cluster 1)**, **Illinois (87 % cluster 1)** and **Texas (90 % cluster 1)**. They lie near the boundary between the groups.
* GMM silhouette = 0.561 and BIC = 287.8.

**Decision.** Treat Missouri, Illinois and Texas as **boundary states** in the report – they could be monitored or analysed separately rather than being forced into a hard category.

## 4.9 Visualize and compare the clustering results

In [ ]:
palette = sns.color_palette("Set1", K_FINAL)
fig, ax = plt.subplots(1, 2, figsize=(17, 7), sharex=True, sharey=True)

# --- K-Means: hard assignment
for c in range(K_FINAL):
    ax[0].scatter(*P[km_labels == c].T, s=60, color=palette[c], label=f"Cluster {c} (n={np.sum(km_labels == c)})", edgecolor="k", lw=.4)
ax[0].scatter(*km_centers.T, marker="X", s=260, color="black", label="Centroid")
ax[0].set_title(f"K-Means (k={K_FINAL}) — hard assignment")

# --- GMM: marker size shrinks as the model becomes less certain
for c in range(K_FINAL):
    mask = gmm_labels == c
    ax[1].scatter(*P[mask].T, s=30 + 150 * gmm_proba.confidence.values[mask], color=palette[c], alpha=.8,
                  label=f"Cluster {c} (n={mask.sum()})", edgecolor="k", lw=.4)
ax[1].set_title(f"GMM (k={K_FINAL}, {GMM_COV}) — probabilistic (bigger marker = more certain)")

for a, lab in zip(ax, [km_labels, gmm_labels]):
    for st, (x_, y_) in zip(arrests.index, P): a.annotate(st, (x_, y_), fontsize=6.5, alpha=.75, xytext=(3, 3), textcoords="offset points")
    a.axhline(0, color="grey", lw=.5); a.axvline(0, color="grey", lw=.5); a.legend(loc="lower left")
    a.set_xlabel("PC1 – overall violent-crime level"); a.set_ylabel("PC2 – rape vs. murder/assault")
plt.tight_layout(); plt.show()

**What the code does.** Plots both clustering results side by side in the PCA plane with the same colours and state labels. K-Means shows the centroids (✖). In the GMM panel the marker size grows with the model's confidence, so uncertain states appear as small dots.

**Why it matters.** Visual comparison lets stakeholders see *where* the clusters lie and how the two algorithms differ.

**Interpretation.**
* Both algorithms draw essentially the **same boundary**: a split along PC1 (overall crime level), with low-crime states on the left (red) and high-crime states on the right (blue).
* The blue group is not homogeneous on PC2: the **south-eastern states** (Mississippi, North Carolina, Georgia, South Carolina, Louisiana, Alabama) are *murder/assault-heavy* (bottom), whereas states such as **Colorado and California** are *rape-heavy* (top). Crime level groups them together, while PC2 shows different crime profiles within the group.
* **Missouri** is the one visible difference: blue under K-Means but red (and a small, uncertain marker) under GMM.

**Decision.** The two-cluster solution is stable across methods. The next cell quantifies the agreement and describes the clusters in real-world units.

In [ ]:
# ---- Quantitative comparison ------------------------------------------------------
ari = adjusted_rand_score(km_labels, gmm_labels)
metrics = pd.DataFrame({
    "K-Means": {"clusters": K_FINAL, "silhouette": silhouette_score(P, km_labels),  "inertia (SSE)": km.inertia_,  "BIC": np.nan,
                "assignment type": "hard"},
    "GMM":     {"clusters": K_FINAL, "silhouette": silhouette_score(P, gmm_labels), "inertia (SSE)": np.nan, "BIC": gmm.bic(P),
                "assignment type": "soft (probabilities)"},
})
display(metrics)
print(f"Adjusted Rand Index between the two partitions: {ari:.3f}   (1 = identical, 0 = random agreement)")
print("\nAgreement table (rows = K-Means, columns = GMM):")
display(pd.crosstab(pd.Series(km_labels, name="K-Means"), pd.Series(gmm_labels, name="GMM")))
diff = arrests.index[km_labels != gmm_labels]
print("States assigned differently by the two models:", list(diff) if len(diff) else "none")

# ---- Business profile of each cluster in ORIGINAL units (UrbanPop included for context) ----
profile = arrests.assign(KMeans=km_labels).groupby("KMeans").agg(["mean"]).droplevel(1, axis=1)
profile.insert(0, "n_states", np.bincount(km_labels))
print("\nAverage crime profile per K-Means cluster (arrests per 100,000; UrbanPop in %):")
display(profile.round(1))
print("Overall US average:"); display(arrests.mean().round(1).to_frame("mean").T)

# ---- Distribution of each variable per cluster
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8))
for a, col in zip(ax, arrests.columns):
    sns.boxplot(x=km_labels, y=arrests[col], palette=palette, ax=a); a.set(xlabel="K-Means cluster", title=col)
plt.tight_layout(); plt.show()

**What the code does.** Quantifies the comparison: silhouette scores, the Adjusted Rand Index (ARI: 1 = identical partitions, 0 = chance), an agreement table, the list of states on which the two models disagree, and the average crime profile of each cluster in **original units** (including the excluded `UrbanPop`), plus box-plots of each variable by cluster.

**Why it matters.** Metrics show whether the methods agree and by how much; the original-unit profiles translate the statistics into a story a policy audience can use.

**Interpretation.**
* **Agreement is very high:** ARI = **0.92**; the two models differ on **just one state (Missouri)** out of 50. Silhouette scores are nearly equal (K-Means 0.568, GMM 0.561).
* **Cluster profiles (K-Means):**

| | States | Murder | Assault | Rape | UrbanPop |
|---|---|---|---|---|---|
| Cluster 0 – "lower violence" | 30 | 4.9 | 114.4 | 15.9 | 63.6 % |
| Cluster 1 – "higher violence" | 20 | 12.2 | 255.2 | 29.2 | 68.4 % |
| All states | 50 | 7.8 | 170.8 | 21.2 | 65.5 % |

* Cluster 1 has roughly **2.5× the murder rate, 2.2× the assault rate and 1.8× the rape rate** of cluster 0. The box-plots show that the first three variables separate the clusters clearly, whereas `UrbanPop` overlaps almost completely (63.6 % vs. 68.4 %).

**Decision.** (1) The result is robust to the choice of algorithm, so either can be used: **K-Means for a simple label, GMM when confidence matters**. (2) Crime level in this data is not driven by how urban a state is, which validates the earlier decision to exclude `UrbanPop` from the clustering.

## 4.10 Sensitivity check: what if the client wants more segments? *(additional step)*

In [ ]:
K_ALT = K_FINAL + 1
km_alt  = KMeans(K_ALT, n_init=50, random_state=RANDOM_STATE).fit(P)
gmm_alt = GaussianMixture(K_ALT, covariance_type=GMM_COV, n_init=20, random_state=RANDOM_STATE).fit(P)
pal = sns.color_palette("Set1", K_ALT)

fig, ax = plt.subplots(1, 2, figsize=(15, 5.8), sharex=True, sharey=True)
for a, lab, title in [(ax[0], km_alt.labels_, f"K-Means k={K_ALT}"), (ax[1], gmm_alt.predict(P), f"GMM k={K_ALT}")]:
    for c in range(K_ALT): a.scatter(*P[lab == c].T, s=55, color=pal[c], edgecolor="k", lw=.4, label=f"Cluster {c} (n={np.sum(lab == c)})")
    a.set_title(title); a.legend(); a.set_xlabel("PC1"); a.set_ylabel("PC2")
plt.tight_layout(); plt.show()

print(f"Silhouette  K-Means k={K_ALT}: {silhouette_score(P, km_alt.labels_):.3f}   vs  k={K_FINAL}: {silhouette_score(P, km_labels):.3f}")
print(f"GMM BIC     k={K_ALT}: {gmm_alt.bic(P):.1f}   vs  k={K_FINAL}: {gmm.bic(P):.1f}   (lower is better)")
print(f"Agreement of K-Means k={K_ALT} with GMM k={K_ALT} (ARI): {adjusted_rand_score(km_alt.labels_, gmm_alt.predict(P)):.3f}")

**What the code does.** Re-runs both algorithms with one extra cluster (k = 3) to see what a finer segmentation would look like, and compares its quality with the k = 2 solution.

**Why it matters.** Clients often ask for more segments ("can we split the high-crime group?"). We should tell them whether the data support it.

**Interpretation.**
* K-Means silhouette falls from **0.568 (k = 2) to 0.445 (k = 3)** – clusters become less distinct.
* GMM BIC rises from **287.8 to 292.9** – the penalised fit is worse.
* The two algorithms **disagree** on how to form three clusters (ARI = 0.50, versus 0.92 for k = 2): the 3-cluster boundaries are unstable and depend on the algorithm, which is typical when a split is not truly supported by the data. The plots show why: **K-Means** keeps the high-crime states together and instead splits the *low-crime* group in two (14 vs. 17 states), whereas **GMM** keeps one large low-crime group (31 states), carves out a tiny 4-state, rape-heavy group at the top of the plot (Colorado, California and neighbours in PC2) and splits the remaining high-crime states by PC2.

**Decision.** Keep **k = 2** as the official result. If the client wants more detail, the safer way to obtain it is to describe the PC2 "crime-type" gradient (murder/assault-heavy vs. rape-heavy) *within* the higher-violence group, rather than to present a third cluster as a statistically established group.

---
# Step 5 — Conclusions and Recommendations for Stakeholders
*Goal: bring the three client projects together, **evaluate** each model on the measures that matter to its client, **visualize** the results, and translate them into clear, honest recommendations.*

**What this step contains**

| Sub-step | Content | Lab requirement covered |
|---|---|---|
| 5.1 | Executive summary | Overview for non-technical readers |
| 5.2 | Confirm the structure of the three datasets | *Summarize the datasets to confirm their structure* |
| 5.3 | Wine classifier — classification report, accuracy, error analysis | *Evaluate the model with a classification report and accuracy score* |
| 5.4 | Feed recommender — recommendations with similarity scores | *Recommend similar feeds based on similarity scores* |
| 5.5 | Crime clustering — K-Means vs GMM comparison | *Visualize and compare clustering results* |
| 5.6 | Cross-project scorecard | Confidence and readiness of each solution |
| 5.7 | Recommendations per stakeholder | Actionable next moves |
| 5.8 | Limitations, risks and responsible use | Honest boundaries of the work |
| 5.9 | Roadmap and next steps | Path from prototype to production |
| 5.10 | Lab requirements checklist | Traceability to the grading criteria |

Unless stated otherwise, every table and chart in 5.2–5.5 is computed live from the objects created in Steps 1–4.

## 5.1 Executive summary

* **Wine classification (Project 1).** A pipeline of *standardization → PCA (95 % variance, 13 → 10 components) → tuned k-NN* identifies the wine variety from 13 chemical measurements with **93.3 % accuracy on unseen wines (42 of 45)** and an expected **≈ 95 %** in the more rigorous nested cross-validation. All three test errors were Class 1 wines. The model is ready as a **decision-support** tool, not yet as a fully automatic gate.
* **Feed recommendation (Project 2).** Using chick weight as the performance proxy, the engine finds **two pairs of near-interchangeable feeds — casein ↔ sunflower (similarity 0.97) and linseed ↔ soybean (0.96)** — and shows that **horsebean has no close substitute**. The ranking agrees with an independent check on average weight (Spearman ρ = 0.96).
* **Crime pattern analysis (Project 3).** Three independent criteria (elbow, silhouette, BIC) agree that the 50 states form **two natural groups: 30 lower-violence and 20 higher-violence states**. K-Means and GMM agree almost perfectly (ARI = 0.92); GMM additionally flags **Missouri, Illinois and Texas as boundary states**.
* **Bottom line.** All three prototypes work and are explainable. Their weakest points are **small samples** and, for the recommender, a **single performance measure**; these determine the next steps in 5.9.

## 5.2 Confirm the structure of the three datasets

In [ ]:
structure = pd.DataFrame({
    "Wine": {
        "file": "wine.csv", "rows (observations)": wine.shape[0], "columns": wine.shape[1],
        "unit of observation": "one wine sample", "feature columns": wine.drop(columns="target").shape[1],
        "feature type": "continuous chemical measurements", "label / target": f"target – {wine['target'].nunique()} classes",
        "missing values": int(wine.isna().sum().sum()), "duplicate rows": int(wine.duplicated().sum()),
        "used in": "Step 2 – k-NN classification"},
    "Chickwts": {
        "file": "chickwts.csv", "rows (observations)": chick.shape[0], "columns": chick.shape[1],
        "unit of observation": "one chick", "feature columns": 1,
        "feature type": "weight (g) – continuous", "label / target": f"feed – {chick['feed'].nunique()} types (grouping variable)",
        "missing values": int(chick.isna().sum().sum()), "duplicate rows": int(chick.duplicated().sum()),
        "used in": "Step 3 – recommendation system"},
    "USArrests": {
        "file": "USArrests.csv", "rows (observations)": arrests.shape[0], "columns": arrests.shape[1],
        "unit of observation": "one US state (row label)", "feature columns": arrests.shape[1],
        "feature type": "arrest rates per 100,000 + % urban population", "label / target": "none (unsupervised)",
        "missing values": int(arrests.isna().sum().sum()), "duplicate rows": int(arrests.duplicated().sum()),
        "used in": "Step 4 – K-Means and GMM clustering"},
})
display(structure)

# Why standardization was necessary: spread of each dataset's features before scaling
spread = pd.DataFrame({
    "Wine":      [wine.drop(columns="target").std().min(), wine.drop(columns="target").std().max()],
    "Chickwts":  [chick["weight"].std(), chick["weight"].std()],
    "USArrests": [arrests.std().min(), arrests.std().max()],
}, index=["smallest feature std", "largest feature std"]).T
spread["largest / smallest"] = spread.iloc[:, 1] / spread.iloc[:, 0]
print("Feature spread BEFORE standardization:"); display(spread.style.format({"smallest feature std": "{:.2f}", "largest feature std": "{:.1f}", "largest / smallest": "{:,.0f}x"}))

print("Chickwts — chicks and weight per feed:"); display(feed_summary.round(1).T)
print("Wine — class distribution:"); display(wine["target"].value_counts().sort_index().to_frame("n_wines").assign(share=lambda d: (d.n_wines / d.n_wines.sum()).round(3)).T)

**What the code does.** Builds one reference table that describes each dataset (size, unit of observation, feature type, label, missing values, duplicates, and where it is used), shows how widely the feature scales differ *before* standardization, and recaps the class balance (wine) and group sizes (chickwts).

**Why it matters.** Before any result is communicated, the reader must be able to verify what data stand behind it. This table is the "ID card" of each project and confirms that the structure assumed by the models is the structure of the data.

**Interpretation.**
* **Wine:** 178 wines × 14 columns (13 chemical features + the 3-class target), no missing values, no duplicates; classes are reasonably balanced (33 % / 40 % / 27 %).
* **Chickwts:** 71 chicks, one numeric feature (weight) and one grouping variable (6 feeds with 10–14 chicks each); no missing values; one duplicated row, which is kept because two different chicks can legitimately weigh the same (Step 1.2).
* **USArrests:** 50 states × 4 numeric variables, no missing values, no label, hence suitable for unsupervised learning.
* **Scale differences:** wine features differ by a factor of about **2,530** (proline vs. nonflavanoid phenols) and USArrests by about **19**, which is why standardization was essential for k-NN, PCA and K-Means. Chickwts has a single feature (ratio 1×), so scaling only changes its unit.

**Decision.** The structure of all three datasets is confirmed, the data are fit for modelling, and no rows or columns had to be removed. The sample sizes (178 / 71 / 50) are small, which is carried into the limitations in 5.8.

## 5.3 Wine classifier — classification report, accuracy and error analysis

In [ ]:
from scipy.stats import binomtest

# ---- (a) Classification report and accuracy on the 45 held-out wines --------------------
acc  = accuracy_score(y_test, y_pred)
n_ok = int((y_pred == y_test).sum())
rep  = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
per_class = pd.DataFrame(rep).T.loc[class_names + ["macro avg", "weighted avg"], ["precision", "recall", "f1-score", "support"]]
lo, hi = binomtest(n_ok, len(y_test)).proportion_ci(confidence_level=0.95, method="exact")
print(f"TEST ACCURACY = {acc:.3f}  ({n_ok} of {len(y_test)} wines)   |   95 % confidence interval: {lo:.3f} – {hi:.3f}")
display(per_class.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1-score": "{:.3f}", "support": "{:.0f}"}))

# ---- (b) Visual: precision / recall / F1 per class ----------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
per_class.loc[class_names, ["precision", "recall", "f1-score"]].plot.bar(ax=ax[0], rot=0, width=.75)
ax[0].axhline(acc, ls="--", color="k", lw=1, label=f"overall accuracy = {acc:.3f}")
ax[0].set(ylim=(0.7, 1.02), ylabel="Score", title="Per-class performance on the test set")
ax[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=4, frameon=False)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=class_names, ax=ax[1], colorbar=False)
ax[1].set_title("Confusion matrix (3 errors)"); ax[1].grid(False)
plt.tight_layout(); plt.show()

# ---- (c) How reliable is that number?  All estimates in one table --------------------------
estimates = pd.DataFrame({
    "estimate": ["5-fold CV on training set (tuned)", "Hold-out test set (45 wines)", "Nested CV on all 178 wines"],
    "accuracy": [grid.best_score_, acc, nested.mean()],
    "comment":  ["used for tuning, slightly optimistic", "single small hold-out; 1 wine = 2.2 points", f"most reliable (± {nested.std():.3f})"]})
display(estimates.style.format({"accuracy": "{:.3f}"}).hide(axis="index"))
display(baseline.style.format("{:.3f}"))

**What the code does.** Re-evaluates the final wine classifier on the 45 unseen test wines: accuracy with a 95 % confidence interval, the classification report (precision, recall, F1 per variety), a bar chart of those metrics, the confusion matrix, and a table placing all accuracy estimates (tuned CV, hold-out, nested CV) and the baseline pipelines side by side.

**Why it matters.** Accuracy answers *"how often is the model right?"*, but stakeholders also need to know **which varieties are confused and how much the number can be trusted**. Precision tells us how reliable a prediction of a given variety is; recall tells us how many wines of a variety are found.

**Interpretation.**
* **Test accuracy = 93.3 % (42 of 45 wines).** Because only 45 wines were tested, the 95 % confidence interval is wide: **81.7 % – 98.6 %**.
* **Class 0:** precision 0.938, recall 1.000, F1 0.968 – every Class 0 wine was found; one wine of another class was wrongly labelled Class 0.
* **Class 1:** precision 1.000, recall **0.833**, F1 0.909 – when the model says "Class 1" it is always right, but 3 of 18 true Class 1 wines were missed (1 → Class 0, 2 → Class 2). This is the **weakest point**.
* **Class 2:** precision **0.857**, recall 1.000, F1 0.923 – all Class 2 wines were found, but 2 of the 14 wines labelled "Class 2" were actually Class 1.
* Macro-average and weighted-average F1 are both ≈ 0.93, so performance is balanced; **Class 0 and Class 2 are never confused with each other** – the model's errors are only "borderline Class 1" cases.
* **Estimates in context:** tuned CV 97.8 % (slightly optimistic because it was used for tuning), hold-out 93.3 % (small sample) and **nested CV 95.5 % ± 1.4 % (most reliable)**. Standardization is the largest single improvement (raw k-NN: 66 % CV), PCA adds a little, and tuning improves CV accuracy further.

**Decision.** Quote **≈ 95 % expected accuracy** (nested CV) to the client with the 93.3 % hold-out result as supporting evidence, and describe the main weakness precisely: *Class 1 wines near the boundary of Class 0 or Class 2 are sometimes misclassified.* The next cell tests whether the model's own confidence can flag those wines.

In [ ]:
# ---- (d) Can model confidence tell us WHICH wines to send for manual checking? ------------
proba   = final_model.predict_proba(X_test)                    # weighted neighbour votes
conf    = proba.max(axis=1)
correct = (y_pred == y_test)

print("Confidence of the correct predictions :", np.round(np.sort(conf[correct])[:5], 2), "... (5 lowest)")
print("Confidence of the 3 wrong predictions :", np.round(np.sort(conf[~correct]), 2))

rows = []
for thr in [0.0, 0.60, 0.70, 0.80, 0.90, 0.99]:
    keep = conf >= thr
    rows.append({"minimum confidence to auto-accept": thr,
                 "wines auto-accepted": f"{keep.sum()} of {len(conf)} ({keep.mean():.0%})",
                 "accuracy on auto-accepted": correct[keep].mean(),
                 "sent to manual check": int((~keep).sum()),
                 "errors caught by manual check": f"{int((~keep & ~correct).sum())} of {int((~correct).sum())}"})
triage = pd.DataFrame(rows)
display(triage.style.format({"minimum confidence to auto-accept": "{:.2f}", "accuracy on auto-accepted": "{:.3f}"}).hide(axis="index"))

plt.figure(figsize=(8, 3.6))
plt.hist([conf[correct], conf[~correct]], bins=np.linspace(0.3, 1.0, 15), stacked=True, label=["correct", "wrong"])
plt.xlabel("Model confidence (max class probability)"); plt.ylabel("Number of test wines")
plt.title("Wrong predictions concentrate at lower confidence"); plt.legend(); plt.tight_layout(); plt.show()

# ---- (e) Safety check: the threshold above was read off only 45 wines, so repeat it on OUT-OF-FOLD predictions for ALL 178 wines
from sklearn.model_selection import cross_val_predict
oof_proba = cross_val_predict(final_model, X, y, cv=outer, method="predict_proba")
oof_conf, oof_ok = oof_proba.max(axis=1), (oof_proba.argmax(axis=1) == y)
rows = []
for thr in [0.0, 0.70, 0.80, 0.90]:
    keep = oof_conf >= thr
    rows.append({"minimum confidence": thr, "wines auto-accepted": f"{keep.sum()} of {len(y)} ({keep.mean():.0%})",
                 "accuracy on auto-accepted": oof_ok[keep].mean(), "sent to manual check": int((~keep).sum()),
                 "errors caught": f"{int((~keep & ~oof_ok).sum())} of {int((~oof_ok).sum())}"})
print(f"Out-of-fold check on all {len(y)} wines (overall accuracy {oof_ok.mean():.3f}):")
display(pd.DataFrame(rows).style.format({"minimum confidence": "{:.2f}", "accuracy on auto-accepted": "{:.3f}"}).hide(axis="index"))

**What the code does.** (d) Looks at the model's confidence for each test prediction (the weighted share of neighbour votes for the winning class) and shows what would happen if low-confidence wines were sent to a human check. (e) Because that threshold was read off only 45 wines, it **repeats the same analysis on out-of-fold predictions for all 178 wines**, a much more reliable sample.

**Why it matters.** A classifier that knows *when it is unsure* can be used safely in a business process: confident predictions are automated, and doubtful ones go to an expert. This turns an imperfect model into a dependable workflow.

**Interpretation.**
* **Test set (45 wines):** the three wrong predictions had confidence of only 0.67, 0.73 and 0.78, lower than almost all correct ones. With a threshold of 0.80, 40 wines (89 %) would be auto-accepted at **100 %** accuracy, and the 5 sent for review would contain all 3 errors. *However, this threshold was chosen after seeing these 45 wines, so the result is optimistic.*
* **All 178 wines (out-of-fold, more honest):** overall accuracy 94.9 %. At a threshold of 0.80, **163 wines (92 %) are auto-accepted at 98.2 % accuracy**, 15 wines (8 %) go to manual review, and **6 of the 9 errors (two-thirds) are caught**. A threshold of 0.90 gives almost the same accuracy (98.1 %) but sends more wines to review (19).
* In other words, confidence is a **useful but imperfect** warning signal: it catches most, not all, errors.

**Decision.** Recommend a **confidence threshold of about 0.80** for a pilot: roughly 1 in 12 wines would be reviewed manually, and errors among the automatically accepted wines would fall from about 5 % to about 2 %. The 0.80 value should be re-validated on newly collected wines before it is fixed in production.

## 5.4 Feed recommender — recommendations based on similarity scores

In [ ]:
def advice(sim, gap_signed):
    kind = "Direct substitute" if sim >= 0.90 else "Comparable alternative" if sim >= 0.70 else "No close substitute"
    if abs(gap_signed) >= 40:                                   # large expected change in chick weight
        kind += f" – but chicks ~{abs(gap_signed):.0f} g {'heavier (upgrade)' if gap_signed > 0 else 'lighter (downgrade)'}"
    return kind

rows = []
for f in cos_sim.index:
    recs = recommend_similar_feeds(f, top_n=2)
    r1, r2 = recs.iloc[0], recs.iloc[1]
    diff = feed_stats.loc[r1.recommended_feed, "mean_weight_g"] - feed_stats.loc[f, "mean_weight_g"]
    rows.append({"farmer uses": f, "mean weight (g)": feed_stats.loc[f, "mean_weight_g"],
                 "1st recommendation": r1.recommended_feed, "similarity 1": r1.cosine_similarity, "match 1": r1.match,
                 "weight vs current (g)": diff,
                 "2nd recommendation": r2.recommended_feed, "similarity 2": r2.cosine_similarity, "match 2": r2.match,
                 "advice": advice(r1.cosine_similarity, diff)})
rec_table = pd.DataFrame(rows).set_index("farmer uses")
display(rec_table.style.format({"mean weight (g)": "{:.0f}", "similarity 1": "{:.3f}", "similarity 2": "{:.3f}", "weight vs current (g)": "{:+.0f}"}))

# ---- Visual: strength of the best match for each feed, and the raw weight evidence -------------
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
plot_df = rec_table.reset_index().sort_values("similarity 1")
sns.barplot(data=plot_df, y="farmer uses", x="similarity 1", hue="match 1", dodge=False, ax=ax[0])
for i, (s, rec) in enumerate(zip(plot_df["similarity 1"], plot_df["1st recommendation"])):
    ax[0].text(s + 0.01, i, f"→ {rec}", va="center", fontsize=9)
ax[0].axvline(0.90, ls=":", color="k", lw=1); ax[0].axvline(0.70, ls=":", color="k", lw=1)
ax[0].set(xlim=(0, 1.3), xlabel="Cosine similarity of the best recommendation", ylabel="", title="Best substitute for each feed (dotted lines: 0.70 / 0.90)")
ax[0].legend(title="match strength", loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2, frameon=False)

order = feed_stats.sort_values("mean_weight_g").index
ax[1].errorbar(feed_stats.loc[order, "mean_weight_g"], range(len(order)), xerr=feed_stats.loc[order, "sd_g"], fmt="o", capsize=4)
ax[1].set_yticks(range(len(order))); ax[1].set_yticklabels(order)
ax[1].set(xlabel="Mean chick weight (g) ± 1 SD", title="Raw evidence: feeds with overlapping bars are natural substitutes")
plt.tight_layout(); plt.show()

print("Worked example — a farmer who currently uses CASEIN:")
display(recommend_similar_feeds("casein", top_n=3).round(3).style.hide(axis="index"))

**What the code does.** Produces the final recommendation table: for each feed a farmer might currently use, the two most similar feeds with their cosine-similarity score, the strength label (strong ≥ 0.90, moderate 0.70–0.90, weak < 0.70), the expected weight difference in grams and a plain-language piece of advice. The left chart shows how strong the best substitute is for every feed; the right chart shows the raw evidence (mean weight ± 1 SD per feed). A worked example lists the top three recommendations for a casein user.

**Why it matters.** This is the deliverable of Project 2 in the form in which a supply company's sales or advisory team can use it. The raw-data chart lets non-technical readers check that the scores make sense.

**Interpretation.**
* **Direct substitutes (similarity ≥ 0.90):** **casein ↔ sunflower (0.965; mean weights differ by only ≈ 5 g)** and **linseed ↔ soybean (0.957; ≈ 28 g apart)**.
* **Comparable alternatives (0.70–0.90):** meatmeal → soybean (0.873) with casein as the second choice (0.811); and horsebean → linseed (0.822).
* **Horsebean is a special case:** its best match is only moderate, its second (soybean, 0.634) is weak, and even the best substitute gives chicks that are **≈ 59 g heavier**. For horsebean users the recommendation is therefore an *upgrade*, not a like-for-like replacement.
* **Worked example (casein user):** sunflower is a strong match (0.965, ≈ 5 g difference), meatmeal is moderate (0.811, ≈ 47 g lighter), and soybean is weak (0.484, ≈ 77 g lighter), so the match strength corresponds to the actual weight gap.
* The right-hand chart agrees: casein and sunflower overlap almost completely, linseed and soybean overlap strongly, and horsebean stands apart at the bottom.

**Decision.** Present **strong matches as direct substitutes, moderate matches as comparable alternatives, and weak ones as not recommended**; always show the expected weight difference next to the similarity score so that farmers see the practical consequence. Flag horsebean as having no close substitute.

## 5.5 Crime clustering — visualizing and comparing K-Means and GMM

In [ ]:
# ---- (a) Head-to-head comparison -----------------------------------------------------------------
cmp = pd.DataFrame({
    "K-Means": {"clusters": K_FINAL, "states in cluster 0 / 1": " / ".join(map(str, np.bincount(km_labels))),
                "silhouette": silhouette_score(P, km_labels), "model-selection criterion": f"inertia = {km.inertia_:.1f} (elbow at k = 2)",
                "assignment": "hard (one label per state)"},
    "GMM":     {"clusters": K_FINAL, "states in cluster 0 / 1": " / ".join(map(str, np.bincount(gmm_labels))),
                "silhouette": silhouette_score(P, gmm_labels), "model-selection criterion": f"BIC = {gmm.bic(P):.1f} (min at k = 2, {GMM_COV})",
                "assignment": "soft (membership probabilities)"},
})
display(cmp)
print(f"Agreement between the two models: ARI = {adjusted_rand_score(km_labels, gmm_labels):.3f}; "
      f"states labelled differently: {list(arrests.index[km_labels != gmm_labels])}")

# ---- (b) What distinguishes the clusters?  Index vs the national average (100 = average state) --------
prof  = arrests.groupby(km_labels).mean()
index = prof / arrests.mean() * 100
ratio = prof.loc[1] / prof.loc[0]
display(pd.concat([prof.round(1).add_prefix("mean "), ], axis=1))
print("Cluster 1 vs cluster 0 (ratio of means):", ratio.round(2).to_dict())

fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
index.T.rename(columns=lambda c: f"Cluster {c} (n={np.sum(km_labels == c)})").plot.bar(ax=ax[0], rot=0, width=.7)
ax[0].axhline(100, color="k", ls="--", lw=1, label="national average (100)")
ax[0].set(ylabel="Index (100 = average US state)", title="Cluster profiles relative to the national average")
ax[0].set_ylim(0, 195); ax[0].legend(loc="upper left", fontsize=9)

# ---- (c) Boundary states: where GMM is least sure --------------------------------------------------
border = gmm_proba.assign(dist=(gmm_proba["P(cluster 1)"] - 0.5).abs()).sort_values("dist").head(8).sort_values("P(cluster 1)")
ax[1].barh(border.index, border["P(cluster 1)"])
ax[1].axvline(0.5, color="k", ls="--", lw=1)
ax[1].axvline(0.1, color="k", ls=":", lw=1); ax[1].axvline(0.9, color="k", ls=":", lw=1)   # beyond the dotted lines = >= 90 % confident
ax[1].set(xlim=(0, 1), xlabel="GMM probability of the higher-violence cluster (dotted: 10 % / 90 %)", title="The 8 least-certain states under GMM")
plt.tight_layout(); plt.show()

# ---- (d) Most extreme states on the main crime axis (PC1) and the uncertain ones ---------------------
pc1 = pd.Series(P[:, 0], index=arrests.index).sort_values()
extremes = pd.DataFrame({"lowest overall violent crime (PC1)": pc1.index[:5], "highest overall violent crime (PC1)": pc1.index[::-1][:5]})
display(extremes.style.hide(axis="index"))
print("States with GMM membership confidence below 90 % (watch-list):")
display(gmm_proba[gmm_proba.confidence < 0.90][["P(cluster 0)", "P(cluster 1)", "assigned", "confidence"]].round(3))

**What the code does.** (a) Compares K-Means and GMM head-to-head (cluster sizes, silhouette, selection criterion, agreement). (b) Translates the clusters into real-world terms by expressing each cluster's average crime rates as an **index relative to the national average (100)**. (c) Shows the eight states where GMM is least certain. (d) Lists the most extreme states on the main crime axis and the GMM watch-list.

**Why it matters.** Cluster numbers mean nothing to policy makers until they are described in terms they know. The comparison also shows how much the conclusion depends on the algorithm chosen.

**Interpretation.**
* **The two algorithms agree almost perfectly:** silhouette 0.568 (K-Means) vs 0.561 (GMM), **ARI = 0.92**, and they differ on **one state only (Missouri)**. Both were independently selected with k = 2 (elbow at k = 2 and BIC minimum at k = 2).
* **Cluster profiles:** the lower-violence cluster (30 states) sits well below the national average on all three crime measures (index ≈ 62–75), whereas the higher-violence cluster (20 states) is far above it (index ≈ 137–156). Cluster 1 has **2.5× the murder rate, 2.2× the assault rate and 1.8× the rape rate** of cluster 0.
* **UrbanPop is almost identical** in both clusters (index 97 vs 104; 63.6 % vs 68.4 %): the clusters are about crime, not about how urban a state is, which supports the feature-selection decision in Step 4.2.
* **Most extreme states:** lowest violent-crime levels – North Dakota, New Hampshire, Vermont, Maine, Iowa; highest – Florida, Nevada, Alaska, California, Michigan.
* **Boundary states (GMM):** Missouri (69 % lower-violence cluster), Illinois (87 % higher-violence) and Texas (90 % higher-violence) have membership confidence below 90 %, so a hard label should be treated with caution. The right-hand chart shows that other states, such as Tennessee and New York, are only just above the 90 % line, while the rest are clearly assigned.

**Decision.** Report **two segments** and use K-Means for the simple label and GMM for the confidence. Treat Missouri, Illinois and Texas as **watch-list states** whose classification could change with slightly different data.

## 5.6 Cross-project scorecard

| | **1. Wine classifier** | **2. Feed recommender** | **3. Crime clustering** |
|---|---|---|---|
| **Technique** | Scaling → PCA → tuned k-NN | Scaling → PCA(1) → feed profiles → cosine similarity | Scaling → feature selection → PCA(2) → K-Means and GMM |
| **Headline result** | 93.3 % test accuracy (42/45); 95.5 % nested CV | Casein ↔ sunflower 0.97, linseed ↔ soybean 0.96; horsebean has no close substitute | 2 clusters (30 / 20 states); K-Means vs GMM ARI 0.92 |
| **How it was validated** | Hold-out test, nested CV, baselines, confidence analysis | Independent mean-weight benchmark (Spearman ρ = 0.96) | Elbow + silhouette + BIC agree; two algorithms agree; k = 3 sensitivity check |
| **Strength of evidence** | **Good** – several consistent estimates | **Moderate** – no ground truth, one measure | **Moderate** – no ground truth, but several independent criteria agree |
| **Main weakness** | Class 1 recall 0.83; small test set (CI 82–99 %) | Only one performance variable (weight); 10–14 chicks per feed | Historical data, only 50 states, k = 3 is unstable |
| **Readiness** | **Pilot with human review** of low-confidence wines | **Prototype** – needs price, nutrition and feed-conversion data | **Exploratory** segmentation framework, not a ranking tool |

## 5.7 Recommendations for stakeholders

### 1. Premium wine distributor
1. **Deploy as decision support, not as an automatic gate.** Auto-accept predictions with confidence ≥ 0.80 (about 92 % of wines in the out-of-fold analysis, with about 98 % accuracy) and route the remaining ≈ 8 % to a manual check; this should remove roughly two-thirds of the misclassifications.
2. **Give special attention to Class 1.** It is the variety that is most often missed (recall 0.83); a prediction of "Class 0" or "Class 2" that comes with low confidence should always be verified.
3. **Track performance in operation.** Record every manual correction; monitor (a) the share of wines sent to review and (b) the error rate among accepted wines. Re-train when either drifts, and in any case when a new vintage, region or variety is added.
4. **Keep the full chemical panel.** The model relies on all 13 measurements (compressed into 10 components); do not drop tests before this is re-validated.
5. **Quantify the cost of an error** (mislabelled stock vs. manual review time) to fine-tune the confidence threshold; the 0.80 value is a starting point, not an optimum.

### 2. Agricultural supply company
1. **Use similarity scores as the language of the recommendation:** ≥ 0.90 → "direct substitute", 0.70–0.90 → "comparable alternative", < 0.70 → "not recommended as a substitute".
2. **Always show the expected weight difference** next to the score (e.g. "linseed gives chicks ≈ 59 g heavier than horsebean") so that farmers can judge the practical impact.
3. **Use substitutes for stock-outs and price pressure:** casein and sunflower can be offered for each other, and linseed and soybean likewise; recommend meatmeal as a flexible middle option.
4. **Position horsebean honestly:** there is no close substitute, so any switch is a performance change (an upgrade in growth), which can be marketed as such.
5. **Before a live launch, enrich the data** with price per kg, protein and energy content, feed-conversion ratio and mortality; a single outcome (weight) is not enough to guarantee that two feeds are truly interchangeable.

### 3. Public-policy research firm
1. **Use the two-segment view as a first screening framework** for comparing states with similar violent-crime profiles (a lower-violence group of 30 and a higher-violence group of 20 states).
2. **Report uncertainty:** show Missouri, Illinois and Texas as boundary states, and use GMM probabilities when a decision depends on a state's exact group.
3. **Look inside the higher-violence group** with the PC2 axis: some states are murder/assault-heavy (mainly in the south-east), others rape-heavy (e.g. Colorado, California). These imply different policy priorities, so the "crime-type" axis can guide differentiated interventions better than a third cluster would.
4. **Do not use `UrbanPop` as an explanation of crime level:** it hardly differs between the two clusters.
5. **Refresh the analysis with current, multi-year data** before any policy recommendation is made (see 5.8).

## 5.8 Limitations, risks and responsible use

| Area | Limitation / risk | How we handled it / what to do |
|---|---|---|
| **Sample size** | 178 wines, 71 chicks and 50 states are small; one wrong wine moves the test accuracy by 2.2 points (95 % CI 82–99 %) | Nested CV, confidence intervals and conservative language; collect more data |
| **Model selection bias** | Tuned CV scores are slightly optimistic because they were used to pick parameters | Reported nested CV and an untouched test set alongside |
| **Confidence threshold** | The 0.80 cut-off was explored on the test set and then re-checked out-of-fold; it caught 6 of 9 errors, not all | Re-validate on new wines; do not promise 100 % error detection |
| **Chickwts – one outcome** | Weight is the only performance measure; PCA to 1 component does not compress anything, and similarity effectively compares weight distributions | Documented in Step 3.2; add richer features before deployment |
| **Chickwts – causality** | Similar weight does not prove similar nutrition, cost or safety | Treat as guidance; check nutritional adequacy and regulations |
| **USArrests – age and scope** | Historical state-level snapshot (1970s); 50 observations; clusters describe association, not causes | Present as a method demonstration; refresh with recent data |
| **USArrests – data meaning** | Arrest rates reflect policing and reporting practices as well as actual crime; state averages hide large differences between regions inside a state | Avoid inferring anything about individual communities or people |
| **USArrests – labels** | Labelling states "high violence" can stigmatise and oversimplify | Use neutral cluster names, always show boundary states and the uncertainty |
| **Feature selection** | Dropping `UrbanPop` was a judgement supported by clustering quality, not an absolute truth | Disclosed in Step 4.2; the other 3-feature subsets scored lower |
| **Unstable finer segments** | k = 3 is not supported (silhouette 0.445, BIC higher, ARI between methods 0.50) | Keep k = 2; describe finer detail through the PC2 gradient |

## 5.9 Roadmap and next steps

| Horizon | Wine classifier | Feed recommender | Crime clustering |
|---|---|---|---|
| **Next 2–4 weeks** | Fix the confidence threshold with the client's own error costs; build a simple report of auto-accepted vs reviewed wines | Collect price, nutrient and feed-conversion data for the six feeds | Obtain recent multi-year crime data for all states |
| **1–3 months** | Benchmark other classifiers (e.g. SVM, logistic regression, random forest) on the same splits and pick the best; collect more wines, especially borderline Class 1 samples | Add these features to the feed profile vectors; test statistically whether feeds differ (ANOVA / Tukey) | Repeat the analysis by year to test whether the two segments are stable; compare with hierarchical and density-based clustering |
| **Production** | Package the full pipeline (scaler → PCA → k-NN), version it, monitor drift and re-train on a schedule | Offer the recommender in the sales tool with similarity scores and weight differences displayed | Publish as a dashboard with uncertainty indicators and neutral wording |

**Overall conclusion.** The three prototypes show that the complete workflow — *preparation → dimensionality reduction → tuned or validated model → evaluation → interpretation* — turns small standard datasets into usable, explainable results. The wine classifier is close to pilot-ready with a human-in-the-loop safeguard, the feed recommender is a convincing prototype that needs richer data, and the crime clustering provides a clear but exploratory two-segment structure whose uncertainty is made visible.

## 5.10 Lab requirements checklist

| Grading criterion | Requirement | Where it is done |
|---|---|---|
| **Dataset preparation** | Load the three datasets; inspect missing values and inconsistencies; standardize; summarize | 1.1 – 1.4, confirmed again in 5.2 |
| **k-NN classification** | Convert labels to numbers; PCA to 95 % variance; GridSearchCV for k and distance metric; train with best parameters | 2.1, 2.3, 2.4, 2.5 |
| **Recommendation system** | Standardize weight; PCA to one component; cosine similarity between feeds | 3.1, 3.2, 3.3 – 3.4 |
| **Clustering (K-Means and GMM)** | Standardize; top-3 features; PCA to 2 components; elbow (inertia) and BIC; K-Means and GMM | 4.1 – 4.8 |
| **Model evaluation and interpretation** | Summarize dataset structure | 1.4 and **5.2** |
| | Recommend similar feeds based on similarity scores | 3.5 and **5.4** |
| | Evaluate the model with a classification report and accuracy score | 2.6 and **5.3** |
| | Visualize and compare clustering results | 4.9 and **5.5** |

---
*End of notebook.*